# COMP5329 — Week 9 Self-Study Material
## Multi-modal Foundation Models

*A standalone companion to the lecture. Covers every topic in `Multi-modal Foundation Model.pdf` (pages 6–66) at depth, independent of the tutorial.*

## Part 0 · Preface & How to Use This Material

### 0.1 What this document is — and isn't

This notebook is a **standalone self-study material**, not a supplement to the Week 9 tutorial. The tutorial only covers Cross-Attention and KV Cache in its 20-minute window; this document covers every topic in the 67-slide lecture at depth, re-explaining Cross-Attention and KV Cache from scratch so you can read this notebook front-to-back without having attended tutorial.

**What you will find here:**
- A full expansion of the lecture's LLM → MLLM → Efficiency → MoE → Robotics flow.
- Ten from-scratch PyTorch code cells for the highest-leverage mechanisms (Cross-Attention, KV Cache, LoRA, MoE gating, Grouped-Query Attention, DPO loss, CLIP contrastive, Flamingo gated cross-attention, Q-Former, SFT mini-example).
- Explicit cross-references to lecture slide pages so you can open the PDF alongside.

**What you will NOT find:**
- A recap of ViT / Mamba / Vision Mamba (assumed complete from Week 7 + Week 8).
- Embedded slide figures (referenced by page number instead, to keep the notebook small).
- Industry-grade implementations (code is minimal and pedagogical, not production).

### 0.2 Prerequisites

This material assumes you have completed:

- **Week 7** — Transformer, self-attention, Q/K/V decomposition, positional encoding.
- **Week 8** — Efficient sequence models: Mamba / Selective State Space Models (S6), Vision Mamba.

If you are shaky on any of these, revisit those weeks' notebooks first. Lecture slides p.2–5 (Quick Review) are a thumbnail reminder; if you understand those slides, you are ready.

### 0.3 Navigation map

| Lecture slide | Notebook section |
|---|---|
| p.6–10 — LLM emerging capabilities, definition | §1.1 |
| p.11–16 — Transformer recap, autoregressive decoding | §1.2 |
| p.17 — Decoder-only vs Encoder-Decoder | §1.3 |
| p.18 — LLM evolution tree | §1.4 |
| p.19 — LLM capabilities | §1.5 |
| p.20 — LLM training pipeline overview | §1.6 |
| p.21–23 — LLM → VLM | §2.1 |
| p.24 — MLLM architecture | §2.2 |
| p.25 — MLLM timeline | §2.3 |
| p.26 — pretrained encoders & LLMs | §2.4 |
| p.27–28 — Connector | §2.5 |
| — Cross-Attention deep dive | §2.6 |
| — CLIP / LLaVA / Flamingo | §2.7 |
| p.29–40 — Non-text generation | §2.8 |
| p.41–46 — LoRA | §3.2 |
| p.47–50 — KV Cache | §3.3 |
| p.51–56 — Mixture of Experts | §4 |
| p.57–66 — Robotics / VLA | §5 |
| — Week 7–9 synthesis | §6 |

---

## Part 1 · Large Language Models *(lecture p.6–20)*

Before we can talk about multi-modal LLMs, we need to nail down three things: (1) what an LLM actually *is*, (2) how it produces text one token at a time, and (3) where in the training pipeline each Week 9 efficiency trick (LoRA, KV Cache, MoE) slots in. Part 1 walks through the full LLM story so that in Part 2 we can stop explaining "the language model" and focus exclusively on "the vision-to-language bridge".

### 1.1 From Emerging Capabilities to the Definition of LLM *(lecture p.6–10)*

The lecturer opens with four screenshots of GPT-4/5 and GPT-4o doing things that nobody explicitly trained them to do (lecture p.7–9): answering a medical-statin question with cited guidelines, diagnosing a yellowing snake plant from a photograph, writing a React app for a "dream tracker" from a single sentence, and rewriting a wedding toast on command. The point of these demonstrations is not to sell you on the models — it is to make a pedagogical claim: **modern LLM capabilities are emergent, not designed.** Nobody wrote code that says "diagnose houseplants"; the capability arose from the combination of scale, data diversity, and next-token prediction.

With that in mind, the textbook definition on p.10 is almost disappointingly simple:

> **A Large Language Model is a large neural network trained on vast amounts of text data to understand, predict, and generate natural language.**

Two words in that sentence are carrying all the weight. The first is **large** — the first-order story of the 2020s is that scaling the parameter count (and the dataset behind it) unlocks qualitatively new behaviours, not just incremental accuracy bumps. The second is **generate** — an LLM's primary output is a probability distribution over the next token, and every capability listed above is a consequence of sampling from that distribution autoregressively. If you keep only one fact from §1.1, keep this: an LLM is "a function from a text prompt to a distribution over the next token, run in a loop."


### 1.2 Transformer Recap & Autoregressive Decoding *(lecture p.11–16)*

We assume from Week 7 that you know what a Transformer encoder is: tokenize the input, embed each token, add positional information, then pass through $L$ blocks where each block is a multi-head self-attention layer followed by a feed-forward MLP (with residual connections and LayerNorm). The encoder's output is a sequence of contextualised token embeddings — a "dense representation" that downstream tasks can consume.

The **decoder side** is where generation actually happens. The lecture walks through it on p.14–16 with an explicit "Round 1 / Round N" diagram. Here is what a single decode step does:

1. Take the sequence so far — the prompt plus everything the model has generated up to this point — and embed each token.
2. Pass it through the decoder stack (same $L$-layer Transformer blocks, but with a **causal mask** inside the self-attention so each position can only attend to positions at or before itself).
3. Take the hidden state at the **last** position, apply an output linear layer ("lm_head") to project it onto the vocabulary, and apply softmax.
4. Pick the next token (greedy / sampled — see §1.6.6), append it to the sequence, go to step 1.
5. Stop when the model emits a special `[Terminate]` token (or `<|endoftext|>`, or `<|eot|>`, depending on the model).

The attention formula inside every block is the one you already know:

$$\mathrm{Attention}(Q, K, V) \;=\; \mathrm{softmax}\!\left(\tfrac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

What the lecturer adds on p.16 is the modern framing: **GPT is decoder-only.** There is no separate encoder stack; the prompt is fed directly into the decoder, and the model generates a continuation. That is the architecture of LLaMA, Qwen, GPT-3/4, Mistral, and every headline LLM since 2020. The causal mask is the single mechanism that makes "decoder-only" work — it stops the model from peeking at future tokens during training, which lets the next-token-prediction objective be computed for all positions in a single forward pass.


### 1.3 Decoder-only vs Encoder-Decoder *(lecture p.17)*

Lecture p.17 splits Large Language Models into three architectural families:

- **Encoder-only** (BERT, RoBERTa, DeBERTa). Bidirectional self-attention, trained with masked language modelling (§1.6.3). Strong at *understanding* tasks: classification, NER, extractive QA. Cannot generate text naturally — you would have to bolt a decoder on top. Mostly frozen out of the 2023+ LLM landscape.
- **Decoder-only** (GPT, LLaMA, Qwen, Mistral, Claude, Gemini Nano). Causal self-attention, trained with next-token prediction. The workhorse of modern LLMs: the prompt and the continuation share one unbroken sequence that flows through one stack of layers.
- **Encoder-decoder** (T5, Flan-T5, BART, UL2). The encoder builds a dense representation of the input, the decoder attends to that representation via **cross-attention** (a mechanism we deep-dive in §2.6) while generating its output autoregressively. Still alive in specialised niches like translation and summarisation.

Both decoder-only and encoder-decoder produce tokens one at a time; both are "autoregressive". The difference is whether there is a separate encoder stack doing a full-sequence pass *before* decoding starts.

**Why decoder-only won the 2020s.** Three reasons. First, architectural uniformity — one stack, trained end-to-end with one objective, scales more cleanly than two stacks that must be coordinated. Second, prompting — once you're already a next-token predictor over a single sequence, instructions can simply be prepended to the generation and the model treats them identically to any other context. Third, training efficiency — with causal masking you can compute the loss at every position in parallel, turning what looks like a serial generation task into a fully parallel training step.


### 1.4 Evolution Tree of LLMs *(lecture p.18)*

Lecture p.18 reproduces the "evolutionary tree" figure from Yang et al.'s *Harnessing the Power of LLMs in Practice*. The tree has three main branches, one per architectural family from §1.3:

- **Encoder-only branch (pink)**: BERT → RoBERTa → ALBERT → ELECTRA → DeBERTa. Mostly a 2018–2021 story. By 2023, the branch has essentially stopped growing — not because the models are bad, but because decoder-only LLMs can be prompted to do the same tasks with no fine-tuning.
- **Encoder-decoder branch (green)**: T5 → BART → FLAN → UL2 → Flan-UL2 → ChatGLM. Alive but narrow. T5 and Flan-T5 are still popular open-source backbones because they are compact and handle conditional generation (summarisation, translation, code) cleanly.
- **Decoder-only branch (blue)**: GPT-1 → GPT-2 → GPT-3 → InstructGPT → ChatGPT → GPT-4 alongside Jurassic, PaLM, LaMDA, LLaMA, LLaMA-2, Claude, and Gemini. This is where all the recent action is, and most of this branch continues past the figure's 2023 cutoff.

The figure also colour-codes **open-source** (shaded boxes) versus **closed-source** (open boxes). That distinction matters for Week 9 specifically: when we pick an LLM backbone for a VLM in Part 2, we almost always pick an *open-source* decoder-only model (LLaMA-2, Vicuna, Qwen, Mistral) because we need access to the weights to insert a visual connector. Closed-source models (GPT-4V, Gemini, Claude) are multi-modal too, but their internals are not available for us to instrument.


### 1.5 LLM Capabilities Taxonomy *(lecture p.19)*

The lecture borrows a three-tier split from Minaee et al.'s LLM survey to organise everything an LLM can do:

**Basic capabilities — the ones scaling alone gives you.**
- *Comprehension*: summarisation, multiple-choice QA, boolean QA, reading comprehension, simplification.
- *World knowledge*: Wikipedia QA, factual recall.
- *Multilingual*: translation, cross-lingual NLI, cross-lingual QA.
- *Coding*: function calling, API calling, code completion.

**Emerging capabilities — the ones that surprise researchers when they appear.**
- *Instruction following*: few-shot learning from task definitions, turn-based dialogue.
- *Reasoning*: arithmetic, symbolic, logical, common-sense reasoning.
- *In-context learning*: completion from positive/negative examples, step-by-step solving, symbolic reference resolution.

**Augmented capabilities — the ones that require coupling the LLM to external systems.**
- *Interacting with users*: physical acting (robotics — the theme of Part 5!), virtual acting (UI automation), assignment planning.
- *Tool utilisation*: task decomposition, tool planning, knowledge-base utilisation.
- *Self-improvement*: self-criticism, self-refinement.

**Why this taxonomy matters for Week 9.** Part 2 (MLLMs) is where we extend the "basic" tier to new modalities — a VLM like LLaVA adds visual comprehension to the basic-comprehension box. Part 5 (robotics VLAs) is where we extend the "augmented" tier — an OpenVLA or π₀.₅ turns *physical acting* from a research dream into a deployable behaviour. With that map in mind, the next section opens up the lecture's "How LLMs Are Built" two-panel chart (p.20) so you understand where LoRA (Part 3) and KV Cache (Part 3) and MoE (Part 4) actually fit in an LLM's lifecycle.


### 1.6 LLM Training Pipeline — Deep Dive *(lecture p.20)*

Lecture p.20 presents a single two-panel chart titled *"How LLMs Are Built? (Part 1 / Part 2)"*. Part 1 covers **pre-training**: data cleaning, tokenisation, positional encoding, architecture choice, model pre-training. Part 2 covers **post-training**: fine-tuning, alignment, decoding strategies, cost-effective adaptation. The slide is an index into a huge amount of machinery; this section opens up each box.

The sub-stages we walk through:

1. **§1.6.1** Data cleaning & Tokenisation (BPE / WordPiece / SentencePiece).
2. **§1.6.2** Positional encoding (absolute / relative / RoPE / ALiBi).
3. **§1.6.3** Pre-training objectives (MLM / CLM / NSP).
4. **§1.6.4** Supervised Fine-Tuning — with a code cell.
5. **§1.6.5** Alignment — RLHF and DPO, with a code cell for the DPO loss.
6. **§1.6.6** Decoding strategies (greedy / beam / top-k / top-p).
7. **§1.6.7** A forward pointer to cost-effective training (LoRA in Part 3, MoE in Part 4).

Keep the whole taxonomy in mind. Part 2's MLLM connector training reuses the SFT recipe almost unchanged; Part 3's LoRA is a drop-in replacement for full fine-tuning; Part 4's MoE is an architectural substitute for the MLP block inside each Transformer layer.


#### 1.6.1 Data Cleaning & Tokenisation

**Data cleaning.** Before a token is ever seen, the raw text corpus goes through an aggressive cleaning pipeline: deduplication (near-duplicate paragraphs are thrown out), language identification (you want to keep exactly the languages your tokeniser covers), quality filtering (low-perplexity filters, classifier-based filters against Common Crawl boilerplate), and PII removal (emails, phone numbers, SSNs). This stage does not make it into academic diagrams, but it consumes a majority of practical engineering effort — and an LLM's personality is largely shaped by what is *left out* of the training corpus.

**Tokenisation.** Once the corpus is clean, each document is sliced into integer IDs by a tokeniser. The slide lists three algorithms:

- **Byte-Pair Encoding (BPE)** — used by GPT-2/3/4 and LLaMA. Starts with bytes as the base vocabulary; greedily merges the most frequent adjacent pair, repeats until the vocabulary reaches the target size (32k–256k).
- **WordPiece** — BERT's tokeniser. Same greedy-merge idea, but each merge step maximises the language-model likelihood on the training corpus instead of raw frequency.
- **SentencePiece** — a byte-level unigram model. Does not require pre-tokenisation on whitespace, so it handles languages without spaces (Chinese, Japanese) uniformly. Used by T5, Qwen, Gemma.

**Why this matters for Week 9.** The tokeniser decides whether non-text modalities can be represented as "tokens" at all. In §2.8 (visual autoregressive models), we will see that SEED and VAR treat images as sequences of discrete codes produced by a VQ-GAN — essentially a *visual BPE*. The entire "LLMs generate images" story only works because text tokenisation has the same interface as image tokenisation: discrete ID → embedding → Transformer.


#### 1.6.2 Positional Encoding

Attention is permutation-invariant: if you shuffle the rows of $Q$, $K$, and $V$ in the same way, the output is just a shuffled version of the original. That means a vanilla Transformer has no concept of word order unless we explicitly inject one. Positional encoding is how we do that. The lecture lists four variants:

- **Absolute Positional Embeddings.** The original Transformer. A fixed sinusoidal vector (or a learned vector, as in BERT) is *added* to each token embedding before the first layer. Simple, but extrapolates poorly: a model trained with max length 512 degrades on length 1024.
- **Relative Positional Embeddings.** Introduced in T5 and Shaw et al. Instead of tagging positions absolutely, the model adds a learned bias to the attention logit for pair $(i,j)$ that depends only on $i-j$. This extrapolates better because positions only matter relative to each other.
- **Rotary Positional Embeddings (RoPE).** The dominant choice in 2024 LLMs (LLaMA, Qwen, Mistral, Gemma). RoPE *rotates* the query and key vectors in 2D subspaces by angles proportional to position: $\mathbf{q}_m \mapsto R_m \mathbf{q}_m$, $\mathbf{k}_n \mapsto R_n \mathbf{k}_n$. Because rotations compose, the inner product $\mathbf{q}_m^{\top} R_{n-m} \mathbf{k}_n$ depends only on the relative offset $n-m$ — you get relative-position semantics through a pure modification of Q and K, with no bias term at all.
- **ALiBi (Attention with Linear Biases).** Press et al. Adds a static, linear, *negative* bias to the attention logit proportional to the distance between tokens: far-apart tokens are penalised, near tokens are not. Dead simple, and extrapolates to much longer contexts than it was trained on.

**Why it matters.** Long-context LLMs (128k, 1M tokens) live or die by the positional encoding. RoPE in particular has become a de-facto standard because it composes nicely with Flash Attention and with the YaRN / NTK-aware scaling tricks used for context-window extension.


#### 1.6.3 Pre-training Objectives

Given a tokeniser and an architecture, the pre-training objective is what the model is actually trying to optimise. The lecture lists four:

- **Masked Language Modelling (MLM).** BERT. Randomly mask 15% of tokens in the input and train the model to predict them using **bidirectional** context. Produces strong representations for *understanding* tasks (classification, NER) but weak for generation, because the model never learns to produce tokens left-to-right.
- **Causal Language Modelling (CLM).** GPT. Predict the next token given the past. The workhorse of all modern generative LLMs. Can be computed in parallel at training time thanks to the causal mask.
- **Next Sentence Prediction (NSP).** BERT's auxiliary objective: given two sentences, predict whether the second followed the first in the original corpus. Mostly abandoned after RoBERTa showed it didn't help.
- **Mixture-of-Experts (MoE) routing.** Not a training *objective* in the same sense, but a training *target* that coexists with CLM: the routing network inside an MoE layer is trained alongside the LM loss (often with an auxiliary load-balance loss — see §4.5). We come back to MoE in Part 4.

For the rest of Week 9, assume **CLM** unless stated otherwise. Every LLM used as a VLM backbone in Part 2 (LLaMA, Vicuna, Qwen, Gemma) is CLM-trained.


#### 1.6.4 Supervised Fine-Tuning (SFT)

After pre-training, the model is a *completion engine*: feed it "The capital of France is", it outputs "Paris". What it is *not* is a helpful assistant — ask it "What's the capital of France?" and the base model might continue with another question ("And what's the capital of Germany?") because that is statistically more likely than a direct answer on random web text.

**SFT fixes this.** We take the pre-trained base model, collect a curated dataset of `(instruction, response)` pairs — an instruction like "What's the capital of France?" paired with a response like "Paris." — and continue training with the same CLM objective, but now with the loss computed *only on the response tokens*. The model learns that when it sees an instruction-shaped prefix, it should generate a helpful, concise completion.

**Dataset sizes in practice.** OpenAI's InstructGPT used around 13,000 prompts written by human labelers. Open-source SFT datasets (Alpaca, ShareGPT, Dolly, OpenAssistant) range from 50k to 1M pairs. LLaVA's visual instruction tuning (§2.7.2) uses ~158k GPT-4-generated instruction-response pairs — exactly the same recipe, just with images attached.

**Limitations of SFT alone.** SFT teaches the model *what good responses look like*, but it does not teach the model *what bad responses to avoid*. A pure SFT model will still confidently hallucinate, still comply with harmful requests, and still prefer verbose answers when concise ones are better. Teaching preferences — "A is better than B" — is what **alignment** (§1.6.5) is for.

The code cell below shows the entire SFT loss in 20 lines. It is just cross-entropy, masked to the response span.


In [ ]:
# === Part 1.6.4: SFT mini-example ===
# Demonstrates that "SFT" is really just CLM fine-tuning on (instruction, response) pairs.
# No real model — we use a tiny embedding + linear head, but the loss shape is what matters.
import torch
import torch.nn as nn

torch.manual_seed(0)

vocab_size, d_model = 20, 8
# Toy "instruction -> response" pairs as token IDs. Response is the label.
pairs = [
    ([1, 2, 3], [4, 5]),
    ([6, 7],    [8, 9, 10]),
]

class TinyLM(nn.Module):
    def __init__(self, V, D):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.head = nn.Linear(D, V)
    def forward(self, ids):
        return self.head(self.emb(ids))

model = TinyLM(vocab_size, d_model)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for step in range(20):
    total_loss = 0.0
    for instr, resp in pairs:
        # Feed instruction+response; predict next token at each position.
        full = torch.tensor(instr + resp)
        logits = model(full)                                  # (T, V)
        # Targets: shift left by one; only the RESPONSE positions contribute to loss.
        targets = torch.tensor(instr[1:] + resp + [0])        # pad tail with 0
        mask = torch.zeros_like(targets, dtype=torch.bool)
        mask[len(instr) - 1 : len(instr) - 1 + len(resp)] = True
        loss = nn.functional.cross_entropy(logits[mask], targets[mask])
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    if step % 5 == 0:
        print(f"step {step:2d}  loss={total_loss/len(pairs):.3f}")

print()
print("The SFT objective is just per-position cross-entropy, masked to the response span.")


#### 1.6.5 Alignment — RLHF and DPO

SFT teaches "what good answers look like" from single demonstrations. **Alignment** teaches "which of two answers is better" from preference pairs. The data shape is different: instead of `(instruction, one_response)`, we collect `(instruction, response_A, response_B, preference)` where a human has marked either A or B as the better response.

**RLHF — the original recipe.** The 3-stage InstructGPT pipeline:
1. Start with an SFT model (that is Stage 0).
2. Train a **reward model** $r_\phi(x, y)$ on preference pairs using a Bradley–Terry loss. The reward model predicts a scalar "how good is response $y$ to prompt $x$."
3. Use **PPO** (proximal policy optimisation, an RL algorithm) to fine-tune the SFT model, with the reward model providing rewards and a KL-divergence penalty against the SFT model keeping the policy from drifting too far.

RLHF works, but it is brittle: you need RL infrastructure, the reward model is a separate training pipeline, and PPO is notoriously tricky to stabilise. It is also expensive — the policy, the reward model, and the reference copy of the SFT model all have to be in memory at once.

**DPO — Direct Preference Optimisation (Rafailov et al., 2023).** The insight: if we make a few assumptions (the Bradley–Terry preference model, a KL-regularised RL objective), we can derive a closed-form loss that skips the reward model entirely. The loss compares the log-probability ratios that the trained policy $\pi_\theta$ and the frozen reference policy $\pi_\mathrm{ref}$ assign to the winning response $y_w$ versus the losing response $y_l$:

$$\mathcal{L}_{\mathrm{DPO}}(\theta) \;=\; -\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_\mathrm{ref}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_\mathrm{ref}(y_l \mid x)}\right)$$

Read it backwards: inside the sigmoid is $\beta$ times the *implicit reward gap* between the chosen and rejected responses, where the implicit reward is $\beta \log \pi_\theta / \pi_\mathrm{ref}$. Maximising $\log \sigma(\text{gap})$ with respect to $\theta$ pushes the policy to assign higher relative probability to winning responses.

**Why DPO exploded in 2024.** No reward model. No PPO. No RL infrastructure. You compute per-response log-probabilities under the policy and the reference, plug them into a one-line loss, and train with the same optimiser you already use for SFT. The code below is the entire DPO loss.

**Other variants worth knowing.** IPO (Identity Preference Optimisation) regularises DPO to avoid degenerate solutions. KTO (Kahneman–Tversky Optimisation) uses prospect-theory utility instead of Bradley–Terry, and works with unpaired "like/dislike" data. The lecture names DPO specifically, and DPO is the algorithm we code up here.


In [ ]:
# === Part 1.6.5: DPO loss from scratch ===
import torch
import torch.nn.functional as F

def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """
    Direct Preference Optimization loss (Rafailov et al., 2023).

    Each *_logps argument is a (batch,) tensor of summed log-probabilities
    of a response under the respective model.
    """
    pi_logratios  = policy_chosen_logps - policy_rejected_logps
    ref_logratios = ref_chosen_logps   - ref_rejected_logps
    logits = beta * (pi_logratios - ref_logratios)
    losses = -F.logsigmoid(logits)
    return losses.mean(), logits

# --- synthetic test: policy already prefers chosen over rejected ---
torch.manual_seed(0)
B = 4
policy_chosen   = torch.tensor([-1.0, -0.5, -2.0, -1.2])
policy_rejected = torch.tensor([-2.0, -1.5, -3.0, -2.2])
ref_chosen      = torch.tensor([-1.5, -1.0, -2.5, -1.7])   # neutral reference
ref_rejected    = torch.tensor([-1.5, -1.0, -2.5, -1.7])

loss, logits = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)
print(f"DPO loss:              {loss.item():.4f}")
print(f"DPO implicit rewards:  {logits.tolist()}")
assert loss.item() > 0, "Cross-entropy-style loss must be positive"
assert (logits > 0).all(), "Policy prefers chosen > rejected, so logits should be positive"
print()
print("DPO loss signs match the preference direction, as expected.")


#### 1.6.6 Decoding Strategies

Once the model is trained (pre-trained + SFT + aligned), we still have one choice left: how to turn the per-step probability distribution into an actual sequence of tokens. The lecture lists four decoding strategies:

- **Greedy**. At each step, pick the argmax. Deterministic, fast, tends to produce repetitive text because it never gets to recover from a locally optimal but globally poor choice.
- **Beam search**. Maintain $k$ partial hypotheses simultaneously; at each step, expand each one with every possible next token, keep the top $k$ by cumulative log-probability. Excellent for machine translation (where there is usually a single "best" translation), disastrous for open-ended generation (it mode-collapses to bland, likely continuations).
- **Top-k sampling**. Truncate the distribution to the top $k$ tokens by probability, renormalise, sample. Random but bounded.
- **Top-p (nucleus) sampling**. Truncate to the *smallest* set of tokens whose cumulative probability exceeds $p$, renormalise, sample. Adaptive: a confident distribution keeps only a handful of tokens, a diffuse distribution keeps many. This is the default strategy for creative generation in almost every production chat model.

**Temperature.** Independent of the truncation scheme above, you can divide the logits by a temperature $T$ before softmax. $T 	o 0$ collapses the distribution onto the argmax (greedy); $T 	o \infty$ flattens it to uniform. Production chat models typically run at $T pprox 0.7$ and $p pprox 0.9$.

**Why this matters for Week 9.** KV Cache (§3.3) is what makes *any* of these decoding strategies fast in practice — without it, every decode step would re-compute the K and V projections for the entire history. And when we talk about VLAs in Part 5, remember that the VLA outputs are tokens from an extended vocabulary — which means the VLA is subject to exactly the same decoding-strategy choices as a text LLM.


#### 1.6.7 Forward pointer — Cost-Effective Training

The lecture's p.20 Part-2 panel ends with a box labelled *"Cost-Effective Training / Inference, Adaptation & Compression"*. That box lists LoRA, ZeRO, RWKV, quantisation, knowledge distillation, and a few others. We defer the parts of that list that matter for Week 9 to dedicated sections:

- **LoRA** → Part 3.2 (training-time efficiency, low-rank weight deltas).
- **KV Cache** (not on the p.20 slide but absolutely crucial at inference) → Part 3.3.
- **MoE** → Part 4 (architectural efficiency, sparse expert activation).
- **Quantisation** → briefly inside Part 3.3.8 (KV-cache quantisation) and Part 6.2 (whole-model quantisation as a concept).

Knowledge distillation, ZeRO, and RWKV are *out of scope* for this Week 9 material — they belong to a broader "efficient training" story that we do not need for MLLMs specifically.

With the LLM training pipeline understood, Part 2 now extends the pipeline to multi-modality: what changes (and what doesn't) when we want the LLM to read images in addition to text.


## Part 2 · Multi-modal Large Language Models *(lecture p.21–40)*

A multi-modal LLM (MLLM) is an LLM that has learned to read images — and in some cases audio and video — by stitching a **vision encoder** onto a **language model** via a **connector** module. That is the entire story, architecturally: almost every MLLM shipped between 2023 and 2025 fits the same three-box template. The interesting design decisions are *which* vision encoder, *which* LLM, and *how* to wire them together. Everything else is engineering detail.

### 2.1 From LLM to VLM — Motivation *(lecture p.21–23)*

Lecture p.22 draws the motivation as a before/after cartoon. **Before**: a text-only GPT-4 is handed a PDF. It can summarise the text ("this document talks about…"), but when handed the PDF's figures it apologises ("Sorry, I cannot understand"). **After**: GPT-4V — with a vision encoder attached — can read the same figures, pointing at bars and labels, quoting captions, and composing answers that draw on both words and pixels. The upgrade from GPT-4 to GPT-4V looks cosmetic, but it unlocks an entire class of tasks that text-only models cannot touch.

There are two reasons to add vision to an LLM. The **cheap** reason: many real-world tasks (document understanding, visual QA, UI navigation, medical imaging) are inherently multi-modal, and text-only models simply cannot be deployed. The **deep** reason: vision is a **grounding source**. Language in isolation is a house of mirrors — the model can say "the apple is red" without the word "red" being tied to anything. Adding visual supervision forces the model's "red" to correspond to a bundle of pixel patterns, which constrains language use toward the real world and, some argue, improves reasoning on tasks that are ostensibly text-only.

**What MLLMs unlock.** Lecture p.23 shows the cartoon: inputs can be any combination of text, image, audio, and video, and outputs (in the most ambitious MLLMs) can *also* be any combination of these. Multi-modal input is the baseline — every MLLM supports it. Multi-modal *output* (generating images, audio, or actions from a text prompt) is a much newer capability and lives in §2.8 and Part 5.


### 2.2 MLLM Architecture: Four Components *(lecture p.24)*

Lecture p.24 shows the canonical four-component MLLM diagram. Learn this diagram. Every VLM we look at in §2.7 — CLIP, LLaVA, Flamingo — is an instantiation of it.

1. **Modality Encoder (a.k.a. Vision Encoder).** A pre-trained network that turns raw pixels (or audio, or video frames) into a sequence of feature vectors. For images, this is almost always a Vision Transformer — CLIP ViT-L/14, EVA, SigLIP, or InternViT. For audio it is Whisper or HuBERT. The key property: the encoder is usually **frozen** during MLLM training. We leverage representations that were learned at Internet scale and do not want to disturb them.

2. **Multimodal Alignment Module (a.k.a. Connector).** The glue. The vision encoder's output lives in *its* embedding space; the LLM's token embeddings live in a *different* space. The connector bridges the two. This is the module that gets the most design attention across papers (§2.5), and it is where the real architectural creativity in MLLMs lives.

3. **LLM Backbone.** A standard decoder-only LLM (LLaMA-2, Vicuna, Qwen, Mistral). The backbone consumes the aligned visual tokens *as if they were text tokens*, runs the usual Transformer stack over them alongside any actual text tokens in the prompt, and produces text output. The LLM is sometimes frozen, sometimes fine-tuned, sometimes fine-tuned with LoRA — this is a training-time choice, not an architectural one.

4. **Generator (optional).** Only present when the MLLM needs to output non-text modalities. Without a generator, the MLLM can only produce text (which is fine for visual QA, captioning, and instruction-following — the majority of 2023 VLMs). With a generator, the MLLM can produce images, audio, or actions. We cover this in §2.8 for images and in Part 5 for robot actions.

**Key insight.** The first three components are what *every* MLLM has. The **design choices** inside each — which vision encoder, which connector style, which LLM, frozen or tuned — are what distinguishes CLIP (contrastive) from LLaVA (token-level MLP) from Flamingo (feature-level gated cross-attention). Keep that in mind as we walk through §2.5's connector taxonomy and §2.7's three case studies.


### 2.3 Development Timeline of MLLMs *(lecture p.25)*

Lecture p.25 reproduces the MLLM timeline from Yin et al.'s *A Survey on Multimodal Large Language Models*. Rather than catalogue every box, note the major milestones:

- **Jan 2023** — Flamingo (DeepMind), Kosmos-1 (Microsoft), VIMA. First wave of "serious" MLLMs; Flamingo introduces gated cross-attention (§2.7.3).
- **Apr 2023** — BLIP-2 (Salesforce) introduces Q-Former (§2.5.2); LLaVA and MiniGPT-4 show that a simple projection between CLIP and a frozen LLM is enough.
- **Aug–Sep 2023** — GPT-4V, Qwen-VL, CogVLM, Kosmos-2.
- **Oct–Dec 2023** — LLaVA-1.5, Video-LLaVA, CogVLM-17B, Gemini.
- **Jan–Mar 2024** — MM1 (Apple), Qwen-VL-Max, MoE-LLaVA (§4.7).

**Observation.** Most open-source MLLMs from 2023–2024 use one of three connector styles — token-level MLP (LLaVA lineage), Q-Former (BLIP-2 lineage), or gated cross-attention (Flamingo lineage). We classify those three styles next.


### 2.4 Pretrained Vision Encoders & Open-Sourced LLMs *(lecture p.26)*

Lecture p.26 gives two reference tables — one for vision encoders, one for LLMs — that are worth internalising because you will see these names everywhere in Part 2 and Part 5.

**Vision encoders commonly used in MLLMs:**

| Variant | Pre-training corpus | Resolution | Samples seen (B) | Params (M) |
|---|---|---:|---:|---:|
| OpenCLIP-ConvNext-L | LAION-2B | 320 | 29 | 197 |
| CLIP-ViT-L/14 | OpenAI WIT | 224 / 336 | 13 | 304 |
| EVA-CLIP-ViT-G/14 | LAION-2B, COYO-700M | 224 | 11 | 1000 |
| OpenCLIP-ViT-G/14 | LAION-2B | 224 | 34 | 1013 |
| OpenCLIP-ViT-bigG/14 | LAION-2B | 224 | 34 | 1845 |
| InternViT-6B | multiple datasets | 448 | – | 5540 |

Why these specifically: **CLIP ViT-L/14** is the default for most VLMs because its features come "language-aligned" from contrastive pre-training. **EVA** adds more training data and tends to outperform CLIP at the same parameter count. **InternViT-6B** is the choice when you need native high-resolution input, which matters for document-understanding VLMs.

**Open-source LLMs commonly used as MLLM backbones:**

| Model | Release | Pre-train data | Params (B) | Languages | Architecture |
|---|---|---|---|---|---|
| Flan-T5-XL/XXL | Oct 2022 | – | 3 / 11 | En, Fr, De | Encoder-decoder |
| LLaMA | Feb 2023 | 1.4T tokens | 7 / 13 / 33 / 65 | En | Causal decoder |
| Vicuna | Mar 2023 | 1.4T tokens | 7 / 13 / 33 | En | Causal decoder |
| LLaMA-2 | Jul 2023 | 2T tokens | 7 / 13 / 70 | En | Causal decoder |
| Qwen | Sep 2023 | 3T tokens | 1.8 / 7 / 14 / 72 | En, Zh | Causal decoder |
| LLaMA-3 | Apr 2024 | 15T tokens | 8 / 70 / 405 | En, Fr, De, … | Causal decoder |

**Why LLaMA dominates.** Open weights with a permissive-enough licence for research, strong benchmark performance, and a vibrant community that ships fine-tunes (Vicuna, Alpaca, WizardLM). Most of the VLMs in §2.7 — including LLaVA — use LLaMA or Vicuna as the backbone.


### 2.5 Connector — Modality Interface *(lecture p.27–28)*

The connector is the module that turns the vision encoder's output (a sequence of visual features in the encoder's embedding space) into something the LLM can consume (tokens in the LLM's embedding space). Lecture p.27 splits connectors into two top-level families:

- **Learnable connector.** A trainable module that maps visual features to LLM tokens. Trained end-to-end (or with some parts frozen). More accurate, requires gradient descent.
- **Expert-to-text translation.** No learnable connector at all. Run a frozen perception tool (image captioner, dense captioner, OCR, object detector) and feed its *text* output to the LLM. Zero training, but lossy — the captioner might miss details that the LLM would otherwise have been able to reason over.

The learnable-connector family further splits into **token-level fusion** (§2.5.2) and **feature-level fusion** (§2.5.3). All three patterns are used in production MLLMs as of 2025.


#### 2.5.1 Expert-to-Text Translation

Concrete example (lecture p.27, right panel): **VideoChat-Text.** A video arrives. A suite of frozen perception experts — InternVideo for clip-level captions, Whisper for transcribed audio, GRiT for dense region captions, T5 for narrative summarisation — each emit a textual description. These descriptions are concatenated and fed to an LLM (ChatGPT, LLaMA, Vicuna, MOSS) as a long text prompt. The LLM never sees raw pixels; it sees a *text dossier* about the video.

**Pros.** No training needed — just pick experts off the shelf. Easy to swap experts when better ones come along. Trivially extensible to new modalities (add an audio expert, add a 3D expert).

**Cons.** Lossy. If the captioner missed an object in the frame, that object is *gone* forever — the LLM cannot "look harder" because it does not have pixels to look at. Also brittle: a captioner's hallucination becomes ground truth for the LLM.

Expert-to-text translation is underrated as a baseline. If you need a multi-modal pipeline and cannot afford to train anything, this approach ships in an afternoon.


#### 2.5.2 Learnable Connector — Token-Level Fusion

Token-level fusion turns visual features into "visual tokens" that are concatenated with the text tokens, and the whole mixed sequence is fed to the LLM as a single input. The LLM does the vision-language reasoning via its own self-attention — no new attention mechanism is added.

**Projection / MLP (LLaVA style).** Dead simple. Let the vision encoder produce $M$ visual features, each of dimension $d_v$. Apply a 2-layer MLP that maps each $d_v$-dimensional feature to a $d_\text{LLM}$-dimensional "visual token". Concatenate the $M$ visual tokens with the text tokens from the instruction, prepend a special `[IMG]` separator, and hand the whole thing to the LLM. That is the **entire** LLaVA connector — about 20 lines of code.

Why it works. Modern LLMs are already good at attending across heterogeneous tokens in a long sequence. If the visual tokens live in the LLM's embedding space (thanks to the MLP), the LLM's self-attention layers automatically learn to "look at" them the same way they look at text tokens. No special machinery required.

**Q-Former (BLIP-2 style).** The problem with pure projection: at 224×224 resolution with a 14-patch ViT, you have 256 visual tokens. At 336×336 with LLaVA-1.5 you have 576. That is a *lot* of tokens to prepend to every instruction — and per-step attention is quadratic in sequence length. Q-Former addresses this with a clever trick: introduce $N_q$ **learnable query vectors** (typically 32) that cross-attend into the frozen image features. The queries' outputs become the visual tokens.

The magic of Q-Former: no matter what the input resolution is, the output is always exactly $N_q = 32$ visual tokens. You can run Q-Former on a 224×224 image or a 672×672 image and get the same 32 tokens out, because the queries attend *across* whatever set of image features exist. This gives the LLM a fixed, short visual prefix regardless of input size — huge savings at inference time.

The code cell below implements a minimal Q-Former forward pass. Notice how `self.queries` is the only new parameter that really carries the "learnable" weight — the cross-attention module itself is standard PyTorch MHA.


In [ ]:
# === Part 2.5.2: Q-Former minimal forward ===
# Shows how learnable queries turn a variable-length image-feature sequence
# into a fixed-length sequence of "visual tokens" via cross-attention.
import torch
import torch.nn as nn

torch.manual_seed(0)

class QFormerLite(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_queries=8):
        super().__init__()
        # The "learnable queries" — this is the whole trick of Q-Former.
        self.queries = nn.Parameter(torch.randn(n_queries, d_model) * 0.02)
        # Standard MHA where Q comes from self.queries and K/V come from image features.
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, image_features):
        # image_features: (B, M, D) — M can vary across runs
        B, M, D = image_features.shape
        q = self.queries.unsqueeze(0).expand(B, -1, -1)          # (B, Nq, D)
        out, attn = self.cross_attn(q, image_features, image_features)
        return self.norm(out + q), attn                          # (B, Nq, D)

qformer = QFormerLite(d_model=64, n_heads=4, n_queries=8)
for M in [49, 196, 576]:
    img_feats = torch.randn(2, M, 64)
    out, attn = qformer(img_feats)
    print(f"image tokens in: {img_feats.shape}  ->  visual tokens out: {out.shape}")
    assert out.shape == (2, 8, 64), "Q-Former output must be fixed-length"

print()
print("Q-Former output is always (B, 8, 64) regardless of M. This is why BLIP-2 uses it.")


#### 2.5.3 Learnable Connector — Feature-Level Fusion

Feature-level fusion takes a different philosophy. Instead of turning visual features into LLM tokens and letting the LLM's own self-attention do the work, we **inject** visual features directly into the LLM's hidden states at specific layers, via new modules wired into the backbone.

**Flamingo (DeepMind, 2022).** Insert a new *gated cross-attention* layer between every few LLM layers. In each of these new layers, text tokens play the role of queries and image features play the role of keys/values — the text "reads" from the image via cross-attention. The innovation is the **gate**: the output of the cross-attention is multiplied by $\tanh(\alpha)$, where $\alpha$ is a *learnable scalar* initialised to zero. At step 0 of training, $\tanh(0) = 0$, so the entire cross-attention layer is a *no-op* and the frozen LLM runs unchanged. As training proceeds, $\alpha$ grows, the gate opens, and the visual signal flows. This is the cleanest recipe for "safely fine-tune a frozen backbone without corrupting the pretrained weights." We implement it in §2.7.3.

**CogVLM.** Every Transformer layer is duplicated: one set of MLP + QKV weights for text tokens, a second set of MLP + QKV weights for image tokens. The attention layer itself is shared — so text and image tokens can attend to each other within each block. The new image weights are trainable; the original text weights can be kept frozen. This preserves the LLM's text capabilities perfectly while adding a "visual expert" on the side.

**LLaMA-Adapter.** A small number of learnable "adapter tokens" are prepended to the text sequence at a subset of Transformer layers. These adapter tokens attend to image features (via zero-initialised attention weights) and then mix visual information into the text stream through the self-attention of those layers. Very parameter-efficient — the adapter adds only a few million trainable parameters on top of a 7B model.

**Token-level vs feature-level trade-off.** Token-level is simpler and uses only one attention mechanism (the LLM's own self-attention), but it lengthens every input sequence by the number of visual tokens. Feature-level keeps sequence length unchanged and can be more parameter-efficient (you can tune just the new injected modules), but it adds architectural complexity. The 2024 trend in open-source VLMs leans token-level (LLaVA, BLIP-2) because simplicity wins when compute is not the bottleneck; the 2022–2023 trend leaned feature-level (Flamingo, CogVLM) because the backbones were expensive to retrain.


### 2.6 Cross-Attention Deep Dive

Cross-attention is the single mechanism that lets one sequence *read* information from another sequence. Once you internalise it, every VLM architecture in §2.7 becomes "just a cross-attention with different labels on the arrows." Spend time on this section — it is the most reused primitive in Week 9.

**Self-attention vs cross-attention — the whole difference.** In standard self-attention (Week 7), we take *one* sequence $X \in \mathbb{R}^{B \times T \times d}$ and project it three ways: $Q = X W_Q$, $K = X W_K$, $V = X W_V$. All three tensors come from the same $X$. The attention output has the same sequence length $T$ as the input.

In cross-attention, $Q$ comes from one sequence and $K, V$ come from *another* sequence:

$$Q = X_\text{text} W_Q, \quad K = X_\text{image} W_K, \quad V = X_\text{image} W_V$$

That is the *only* architectural difference. The attention formula is unchanged:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\tfrac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

But the interpretation is completely different: now each text token "queries" every image patch and gets a weighted-average patch feature in return.

**Why "Q from text, K/V from image"?** This is the natural pattern for a VLM. The text tokens are asking questions ("what object is at the top-left?" "what colour is the cup?"); the image patches are providing the answers. The query is the question-asker; K and V are the database being queried. You *could* flip the roles ("Q from image, K/V from text") — it just means the image patches are the ones doing the asking — but in almost every VLM that uses cross-attention, the text is the query.

**Shape derivation — internalise these shapes.** Let $B$ be batch size, $N$ the number of text tokens, $M$ the number of image patches, $d$ the model dimension, $H$ the number of heads, $d_h = d / H$ the per-head dimension.

- Text (query source): $X_\text{text} \in \mathbb{R}^{B \times N \times d}$
- Image (key/value source): $X_\text{image} \in \mathbb{R}^{B \times M \times d}$
- After projection: $Q \in \mathbb{R}^{B \times N \times d}$, $K \in \mathbb{R}^{B \times M \times d}$, $V \in \mathbb{R}^{B \times M \times d}$
- Multi-head reshape: $Q \in \mathbb{R}^{B \times H \times N \times d_h}$, $K \in \mathbb{R}^{B \times H \times M \times d_h}$, $V \in \mathbb{R}^{B \times H \times M \times d_h}$
- Attention scores: $Q K^{\top} \in \mathbb{R}^{B \times H \times N \times M}$ — **rectangular**, not square!
- Softmax over the last dim (the $M$ patches): $\mathrm{attn} \in \mathbb{R}^{B \times H \times N \times M}$
- Weighted sum: $\mathrm{attn} \cdot V \in \mathbb{R}^{B \times H \times N \times d_h}$
- Concatenate heads: output $\in \mathbb{R}^{B \times N \times d}$

**The crucial observation.** The output shape matches the *query* shape, which is the *text* shape. The output has $N$ rows — one per text token. The image has been "read into" the text stream without changing the text length at all. That is exactly what a VLM needs: a way to absorb visual information into the existing text sequence.


In [ ]:
# === Part 2.6: Cross-Attention from scratch ===
# Q from text (N tokens), K/V from image (M patches).
# Attention matrix is RECTANGULAR: (B, n_heads, N, M).
import math
import torch
import torch.nn as nn

class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_text, x_image):
        B, N, D = x_text.shape
        M = x_image.shape[1]

        # Q from TEXT, K/V from IMAGE — the single most important line.
        Q = self.W_q(x_text)    # (B, N, D)
        K = self.W_k(x_image)   # (B, M, D)
        V = self.W_v(x_image)   # (B, M, D)

        # Multi-head reshape: (B, n_heads, seq, d_head)
        Q = Q.view(B, N, self.n_heads, self.d_head).transpose(1, 2)
        K = K.view(B, M, self.n_heads, self.d_head).transpose(1, 2)
        V = V.view(B, M, self.n_heads, self.d_head).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, H, N, M)
        attn   = scores.softmax(dim=-1)
        out    = attn @ V                                          # (B, H, N, d_head)

        out = out.transpose(1, 2).reshape(B, N, D)
        return self.W_o(out), attn

# --- smoke test ---
torch.manual_seed(0)
B, N, M, D, H = 2, 7, 49, 64, 4   # 49 = 7x7 image patches
layer = CrossAttention(D, H)
x_text  = torch.randn(B, N, D)
x_image = torch.randn(B, M, D)
out, attn = layer(x_text, x_image)

assert out.shape  == (B, N, D),    f"out shape wrong: {out.shape}"
assert attn.shape == (B, H, N, M), f"attn shape wrong: {attn.shape}"
print(f"text tokens N={N}, image patches M={M}")
print(f"attn matrix shape: {tuple(attn.shape)}   <- RECTANGULAR (N x M), not square")
print(f"output shape:      {tuple(out.shape)}    <- matches text, not image")
print()
print("Cross-Attention forward verified.")

# --- optional heatmap (first head, first batch) ---
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    heat = attn[0, 0].detach().numpy()
    fig, ax = plt.subplots(figsize=(5, 3))
    im = ax.imshow(heat, aspect="auto")
    ax.set_xlabel(f"image patches (M={M})")
    ax.set_ylabel(f"text tokens (N={N})")
    ax.set_title("Cross-Attention weights (head 0, batch 0)")
    fig.colorbar(im)
    fig.tight_layout()
    # NOTE: in a real notebook we'd call plt.show(); here we keep it headless.
    plt.close(fig)
    print("Heatmap rendered (check the cell output when you run this in Jupyter).")
except ImportError:
    print("(matplotlib not installed — skipping heatmap)")


#### Self-Attention vs Cross-Attention — cheat sheet

| | Self-Attention | Cross-Attention |
|---|---|---|
| **Where does Q come from?** | Same sequence | One sequence (e.g. text) |
| **Where do K, V come from?** | Same sequence | A different sequence (e.g. image) |
| **Attention matrix shape** | $N \times N$ (square) | $N \times M$ (rectangular) |
| **Output length** | Same as input ($N$) | Same as the **query** ($N$) |
| **Typical use in VLMs** | inside the LLM backbone, inside the vision encoder | connector layer (Flamingo); encoder→decoder attention in T5 |

Keep this table in mind for the rest of Part 2. We will see cross-attention used in anger in §2.7.3 (Flamingo), implicitly inside §2.5.2's Q-Former (learnable queries cross-attending to image features), and again in §3.3.2 (when we argue about KV caching in encoder-decoder models).


### 2.7 Three Representative VLMs Compared

Of the hundreds of VLMs in the p.25 timeline, three are historically pivotal and architecturally distinct. **CLIP** (2021) established that contrastive vision-language pre-training works and produced the vision encoder that almost every later VLM reuses. **LLaVA** (2023) showed that a 2-layer MLP is enough to wire a frozen vision encoder to a frozen LLM, and that GPT-4 can generate the instruction-tuning data needed to teach "assistant" behaviour over images. **Flamingo** (2022) introduced gated cross-attention for interleaved vision-language input, and its `tanh` gate is still the textbook recipe for "safely fine-tune a frozen backbone." Each section below walks through one of the three, and §2.7.4 puts them side by side.


#### 2.7.1 CLIP — Contrastive Language-Image Pre-training

**Architecture.** Two separate encoders: a Vision Transformer (ViT-B/32, ViT-L/14, or ViT-H/14 in the larger variants) that takes an image and produces a single pooled embedding; and a text Transformer that takes a caption and produces a single pooled embedding. Both embeddings live in the same $d$-dimensional space after an L2 normalisation at the output. That is all CLIP is — two encoders with one loss.

**Training objective — symmetric InfoNCE.** Take a batch of $B$ (image, caption) pairs. Compute all $B \times B$ similarities between image embeddings and text embeddings. The diagonal entries are the "correct" pairs; the off-diagonals are distractors. Train with cross-entropy to push the diagonal up and the off-diagonals down, *symmetrically*: for each image, the target is its own caption; for each caption, the target is its own image. The loss is the average of the two cross-entropies.

**Training data.** OpenAI's WIT dataset: ~400 million (image, caption) pairs scraped from the web. LAION-2B and LAION-5B (used by OpenCLIP variants) are larger open re-creations.

**Why CLIP matters for VLMs.** The CLIP vision encoder is the **vision encoder** used by LLaVA, BLIP-2, MiniGPT-4, InstructBLIP, and most 2023 VLMs. Its features come out of the encoder already "language-aligned" — a dog patch in the image ends up at the same location in embedding space as the word "dog" in the text encoder. This means downstream VLMs get a head start: they do not need to teach the vision encoder what a dog looks like, because CLIP already did.

The code below is the entire CLIP loss. It is ~15 lines.


In [ ]:
# === Part 2.7.1: CLIP contrastive loss ===
# Symmetric InfoNCE between image and text embeddings.
import torch
import torch.nn.functional as F

def clip_loss(image_emb, text_emb, temperature=0.07):
    """
    image_emb, text_emb: (B, D) embeddings (will be L2-normalised).
    Returns the symmetric cross-entropy over the similarity matrix.
    """
    image_emb = F.normalize(image_emb, dim=-1)
    text_emb  = F.normalize(text_emb,  dim=-1)

    logits = image_emb @ text_emb.t() / temperature   # (B, B)
    labels = torch.arange(image_emb.size(0), device=image_emb.device)

    loss_i2t = F.cross_entropy(logits, labels)        # each image -> its text
    loss_t2i = F.cross_entropy(logits.t(), labels)    # each text  -> its image
    return (loss_i2t + loss_t2i) / 2, logits

# --- smoke test ---
torch.manual_seed(0)
B, D = 4, 16
img = torch.randn(B, D)
txt = img + 0.05 * torch.randn(B, D)   # nearly matched -> loss should be low
loss, _ = clip_loss(img, txt)
print(f"CLIP loss (matched):    {loss.item():.4f}")

txt_scrambled = txt[torch.randperm(B)]  # break the matching
loss_bad, _ = clip_loss(img, txt_scrambled)
print(f"CLIP loss (scrambled):  {loss_bad.item():.4f}")
assert loss.item() < loss_bad.item(), "Matched pairs should give lower contrastive loss"
print()
print("Matched pairs give lower contrastive loss, as expected.")


#### 2.7.2 LLaVA — Visual Instruction Tuning

**Architecture.** CLIP ViT-L/14 (frozen) → 2-layer MLP projection → Vicuna-13B LLM. That is the entire connector: a 2-layer MLP. Each of the 576 visual features (at 336×336 resolution) is mapped by the MLP into a Vicuna token embedding, and the resulting 576 "visual tokens" are prepended to the text tokens of the instruction before being fed to the LLM. Exactly the token-level fusion pattern from §2.5.2.

**Training — two stages.**

*Stage 1: Pre-train the projection.* Freeze both CLIP and Vicuna. Train only the 2-layer MLP on roughly 600k image-caption pairs from CC3M, using next-token prediction over the caption. The goal is to align the visual token embeddings with Vicuna's word embedding space. This takes a few GPU-hours and produces an adapter that speaks Vicuna's embedding language.

*Stage 2: Visual Instruction Tuning.* Unfreeze the LLM (or apply LoRA to it — both variants exist). Train on ~158k instruction-response pairs that were *generated by GPT-4* from COCO images. The instructions look like "What is unusual about this image?" and the responses look like detailed multi-sentence descriptions. This is the stage that teaches the model "assistant behaviour" over visual content. Stage 2 takes about a day on 8 A100 GPUs.

**Why LLaVA is historically important.** Three reasons. First, it showed that a **2-layer MLP** is all the connector you need if the underlying encoders are strong. Every subsequent "minimalist" VLM owes something to this observation. Second, it showed that **GPT-4 can generate the instruction data** — so you don't need thousands of humans writing annotations. Third, it **released open weights** and a reproducible training recipe, which catalysed the open-source VLM community in 2023.

**LLaVA-1.5 (2023) and LLaVA-NeXT (2024)** are incremental refinements: higher resolution, better data, minor architecture tweaks, but the fundamental recipe is unchanged. If you only study one VLM in depth, make it LLaVA.


#### 2.7.3 Flamingo — Gated Cross-Attention for Interleaved Vision-Language

**Architecture.** A frozen vision encoder (NFNet-F6 in the original paper) + a frozen Chinchilla LLM + *trainable gated cross-attention layers* inserted between every few LLM blocks. The vision encoder produces a set of image features; a "Perceiver Resampler" (essentially a Q-Former) compresses them to a fixed number of visual tokens; those visual tokens are the K/V source for the gated cross-attention layers.

**The key innovation — the gate.** Each gated cross-attention layer outputs:

$$\text{out} = x_\text{text} + \tanh(\alpha) \cdot \text{CrossAttn}(Q{=}x_\text{text}, K{=}x_\text{image}, V{=}x_\text{image})$$

where $\alpha$ is a *learnable scalar parameter* initialised to zero. At step 0 of training, $\tanh(0) = 0$, so the cross-attention output is multiplied by zero — the entire gated layer is a **no-op**. This means the Flamingo model at initialisation behaves *exactly* like the original frozen Chinchilla LLM on text-only inputs. Training is then free to slowly open the gate (by learning $\alpha > 0$) without ever destabilising the pretrained weights.

Compare this to "just fine-tune the LLM with cross-attention layers" — if the new cross-attention layers start with random weights, they will inject garbage into the residual stream, and the frozen LLM's behaviour will be corrupted until the random weights settle. The gate trick avoids this entirely.

**Interleaved training data.** Flamingo's other innovation is that its training data is **interleaved**: a single training sample is a sequence like "some text ... [IMG] image tokens ... more text ... [IMG] image tokens ... even more text." This teaches the model to handle multiple images within one context window, which is what you need for in-context few-shot visual reasoning ("here are three labelled examples, now label this one").

**Legacy.** Flamingo has been largely replaced by LLaVA-style token-level fusion for open-source VLMs — the gate trick is elegant but adds architectural complexity that the community often doesn't need. However, the `tanh(α)` gate lives on in almost every "add new modules to a frozen backbone" recipe, including LLaMA-Adapter and various robotics VLAs in Part 5.

The code below implements the gated cross-attention layer. Notice how `attn_gate` is a single scalar parameter initialised to zero.


In [ ]:
# === Part 2.7.3: Flamingo-style gated cross-attention ===
# The critical trick: initialise the gate at 0 so the module is a no-op at step 0.
import math
import torch
import torch.nn as nn

class GatedCrossAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        # Flamingo's gate: start at 0 so tanh(0)=0 -> module is a no-op.
        self.attn_gate = nn.Parameter(torch.zeros(1))

    def forward(self, x_text, x_image):
        B, N, D = x_text.shape
        M = x_image.shape[1]

        Q = self.W_q(x_text).view(B, N, self.n_heads, self.d_head).transpose(1, 2)
        K = self.W_k(x_image).view(B, M, self.n_heads, self.d_head).transpose(1, 2)
        V = self.W_v(x_image).view(B, M, self.n_heads, self.d_head).transpose(1, 2)

        attn = (Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)).softmax(dim=-1)
        out  = (attn @ V).transpose(1, 2).reshape(B, N, D)
        out  = self.W_o(out)

        # Residual + gate
        return x_text + torch.tanh(self.attn_gate) * out

# --- smoke test 1: at init, the layer is identity (pretraining behaviour preserved) ---
torch.manual_seed(0)
layer = GatedCrossAttention(d_model=64, n_heads=4)
x_text  = torch.randn(2, 8, 64)
x_image = torch.randn(2, 49, 64)

out_init = layer(x_text, x_image)
print(f"max |out - x_text| at init (gate=0): {(out_init - x_text).abs().max().item():.6f}")
assert torch.allclose(out_init, x_text, atol=1e-6), "At init, gated cross-attn must equal input"

# --- smoke test 2: opening the gate produces non-trivial output ---
with torch.no_grad():
    layer.attn_gate.fill_(0.5)
out_open = layer(x_text, x_image)
diff = (out_open - x_text).abs().max().item()
print(f"max |out - x_text| after opening gate to 0.5: {diff:.4f}")
assert diff > 0.01, "After opening the gate, output should differ from input"
print()
print("Gate starts at 0 (no-op) and opens as we train it.")


#### 2.7.4 Three VLM Paradigms Compared

| | **CLIP** | **LLaVA** | **Flamingo** |
|---|---|---|---|
| **Connector style** | *(no LLM)* — two aligned encoders | MLP projection (token-level fusion) | Gated cross-attention (feature-level fusion) |
| **What is trainable?** | Both encoders | Stage 1: MLP only. Stage 2: MLP + LLM | Gated cross-attn layers only (backbones frozen) |
| **Training data scale** | 400M image-text pairs (WIT) | 600k captions + 158k instructions | Interleaved web data + captions |
| **Killer feature** | Embeddings aligned across modalities; reusable as a vision encoder | Dead simple; GPT-4-generated instruction data | Safe fine-tuning via $\tanh$ gate; interleaved few-shot |
| **Key paper** | Radford et al. 2021 | Liu et al. 2023 | Alayrac et al. 2022 |
| **Open weights?** | Yes (and OpenCLIP re-release) | Yes | Closed; OpenFlamingo is an open re-implementation |

**How to use this table.** CLIP is not an MLLM in the text-generating sense — it is an embedding model. But it is the *supplier* for almost every downstream MLLM's vision encoder. LLaVA is the modern open-source default when you want the simplest possible recipe. Flamingo's gated cross-attention lives on as the architectural pattern for "add trainable modules to a frozen backbone" — you will see it again in Part 5 when robotics VLAs need to bolt action heads onto pretrained VLMs without disturbing them.


#### The VLM Design Space

Stepping back from the three specific models: designing a new VLM boils down to three orthogonal choices.

**Axis 1 — Vision encoder.** CLIP ViT-L/14 (standard), EVA-CLIP (more data, better performance at the same size), SigLIP (a newer CLIP variant with sigmoid loss, used by PaliGemma in Part 5), InternViT (native high resolution, for document-heavy tasks), DinoV2 (self-supervised, richer spatial features — used in OpenVLA in Part 5). Your choice of encoder largely determines what *kind* of visual details the downstream VLM can reason about.

**Axis 2 — Connector style.** Expert-to-text (training-free, lossy), token-level fusion with MLP (LLaVA), token-level fusion with Q-Former (BLIP-2), feature-level fusion with cross-attention (Flamingo), feature-level fusion with visual experts (CogVLM), feature-level fusion with adapters (LLaMA-Adapter). The choice is a trade-off between simplicity, parameter efficiency, and the amount of visual compute you are willing to spend per forward pass.

**Axis 3 — LLM size + training depth.** Small-and-fully-tuned (LLaVA-1.5 fine-tunes Vicuna-13B), small-and-LoRA-tuned (LLaVA with LoRA — §3.2 forward reference), large-and-frozen (Flamingo uses frozen Chinchilla-70B), or huge-and-MoE (MoE-LLaVA from §4.7). This axis trades inference cost against capability.

Almost every VLM in the timeline can be described by picking one value from each axis. When you read a new VLM paper, start by asking "what did they pick on each of the three axes?" — the paper's novelty usually lives in exactly one of them.


### 2.8 Generating Non-Text Modalities *(lecture p.29–40)*

So far every VLM we have seen (CLIP, LLaVA, Flamingo) is a **multi-modal input, text-only output** system — you feed it an image and a question, it returns text. The lecture's next big theme is *letting MLLMs output non-text modalities*: images, video, and in Part 5, robot actions. There are two strategies:

- **LLMs as Conditioner.** The LLM does not generate pixels itself. It acts as a *planner / router*: it interprets the user's request, decides what the output should look like, and then calls a specialised generative model (usually a diffusion model) that actually produces the pixels. §2.8.1 — DiffusionGPT.
- **LLMs as Generator.** The LLM generates non-text outputs directly, by having a vocabulary that includes *visual tokens* alongside text tokens. Three flavours:
  - **Visual Autoregressive** — treat an image as a sequence of discrete codes (via VQ-GAN) and predict them one-by-one. §2.8.2 — SEED, SEED-X.
  - **Visual Scale Autoregressive** — predict an image as a hierarchy of resolutions, not a flat sequence of patches. §2.8.3 — VAR.
  - **Visual Diffusion** — the LLM backbone does both CLM (for text) and a diffusion objective (for image tokens) simultaneously. §2.8.4 — Transfusion, Show-o.

The common thread: if we can *discretise* the output modality into tokens, we can reuse the LLM's training and inference machinery verbatim. This is the same philosophy as the robot-action tokenisation in Part 5.


#### 2.8.1 LLMs as Conditioner — DiffusionGPT *(lecture p.30–31)*

DiffusionGPT (Qin et al., 2024) takes a clean division-of-labour approach: the LLM handles *understanding and routing*, a diffusion model handles *pixel generation*. Neither one tries to do the other's job.

**The pipeline (lecture p.31):**

1. **Prompt Parse Agent.** The LLM reads the user's prompt ("generate an image of a laughing woman, fashion magazine cover") and parses it into a structured description — style tags, composition, subject, mood.
2. **Tree-of-Thought of Models.** The LLM enumerates candidate diffusion models based on the parsed description. It maintains a *tree* of models organised by style (photo, cinematic, anime, illustration) and traverses it based on the prompt. For the example prompt, the tree might walk down "photo → cinematic → FilmVelvia2".
3. **Model Selection Agent.** Given the candidate models, an LLM-driven agent (possibly guided by human feedback) picks the single best one for this particular prompt.
4. **Prompt Extension + Execution.** The LLM rewrites the user's prompt into a detailed, model-specific caption ("The woman on the magazine cover is laughing joyfully, her eyes twinkling with delight. She is wearing a fashionable outfit..."), then calls the selected diffusion model to generate the image.

**Why this works.** Each stage is something an LLM is already good at: parsing natural language, reasoning over structured options, choosing among alternatives, and rewriting text for a specific downstream consumer. The LLM never generates pixels — that is what diffusion models are good at — and the diffusion model never reasons about the user's intent — that is what LLMs are good at. The two cooperate via a text interface.

**Related: ThinkDiff** is another "LLM as conditioner" system where the LLM generates an intermediate "thought" that conditions the diffusion model more expressively than a raw caption would. Same philosophy, different interface between the two components.

**Take-away.** DiffusionGPT is a *planning* architecture, not a new generative model. If you have access to great text-to-image models but want your system to make smart routing and prompt-rewriting decisions, put an LLM in front — exactly as DiffusionGPT does.


#### 2.8.2 LLMs as Generator — Visual Autoregressive

The alternative to "delegate to diffusion" is to have the LLM itself generate images the same way it generates text: as a sequence of discrete tokens, predicted one at a time, from a shared vocabulary. This requires a **visual vocabulary** — a finite set of discrete codes that can represent any image. Three ingredients make it work:

1. **Visual Vocabulary.** A VQ-GAN (vector-quantised GAN) or VQ-VAE (vector-quantised autoencoder) is trained to compress image patches into discrete codes from a learned codebook of ~8,192 entries. You can think of it as a VQ-BPE for images: an image patch is the "character", the codebook is the "vocabulary", the VQ encoder is the tokeniser, and the VQ decoder reverses the process to turn codes back into pixels. This step is usually done once up front and frozen; the LLM never retrains it.

2. **Unified Learning.** Tokenise text with a normal BPE tokeniser and images with the VQ codebook. The two vocabularies are *disjoint and concatenated* — text token IDs are 0…50000 and visual token IDs are 50001…58192. The LLM sees a single mixed sequence like `[text_tokens] [IMG] [visual_tokens] [/IMG] [more_text_tokens]` and treats visual tokens like any other vocabulary.

3. **Next-Token Prediction.** Train with standard CLM loss over the joint vocabulary. At inference, generate autoregressively — when the model emits `[IMG]` it starts producing visual token IDs; when it emits `[/IMG]` those IDs are passed to the frozen VQ decoder and rendered as pixels.

That is the whole recipe. The LLM doesn't know (and doesn't care) that some of its tokens correspond to pixels — the visual vocabulary is just another part of its vocabulary.


##### SEED (Tencent, 2023) — multimodal autoregression with causal visual tokens

SEED (Ge et al., 2023) is the cleanest instantiation of visual autoregression. Its visual tokeniser (lecture p.34) has five stages:

1. **ViT Encoder** — a pretrained vision encoder (from BLIP-2) that extracts 2D spatial features from the input image. Frozen.
2. **Causal Q-Former** — here is the novelty. A Q-Former variant with a *causal mask* in its self-attention, so that later queries cannot see earlier queries. This turns the 2D spatial feature map into a 1D *causal* sequence — each output token only depends on the tokens before it, mirroring the behaviour of text tokens in an autoregressive LLM. The reason this matters: a standard (non-causal) Q-Former produces tokens that have all-to-all dependencies, which breaks the LLM's causal generation.
3. **VQ Codebook** — discretise the 1D causal features into codes from a learned codebook. Now the image is a sequence of integers, indistinguishable in type from text tokens.
4. **Reverse Q-Former** — at generation time, given visual token IDs, decode them into a conditioning signal for a Stable Diffusion UNet. This is the inverse of step 2.
5. **UNet Decoder (Stable Diffusion)** — frozen pretrained SD. Consumes the reverse Q-Former's conditioning and produces pixels.

**At training time**, the LLM is trained on interleaved sequences that contain both text tokens and visual codes produced by SEED. The single training objective is next-token prediction over the joint vocabulary. The model learns to emit visual token IDs when generating images and text token IDs when generating text.

**At inference time**, the LLM generates autoregressively. When it decides to generate an image, it emits a sequence of visual codes; those codes go through the reverse Q-Former and the Stable Diffusion UNet to produce the actual pixels.

SEED is important mostly as an *architecture*: it showed that a causal Q-Former + VQ codebook is a workable way to give an LLM a visual vocabulary. SEED-X takes the recipe further.


##### SEED-X (Tencent, 2024) — the three-way token objective

SEED-X (Ge et al., 2024) is the deepest example of MLLM generation in the lecture, and it rewards careful reading. The design goal is aggressive: **build a visual tokeniser whose tokens are simultaneously causal, semantic, and generative.** Let's unpack each of those properties and then walk through the training recipe.

**The three properties a SEED-X visual token must have:**

- **Causal.** As in SEED: tokens produced later can only depend on tokens produced earlier. This is a hard architectural requirement for autoregressive use — without it, the LLM's generation is inconsistent with how the tokens were trained.
- **Semantic.** A visual token must align with text meaning. Concretely: when the visual tokens of an image of a dog are fed through a comparable text encoder, the text embedding "a photo of a dog" should be near the visual token embeddings. Why: otherwise the LLM, which reasons in a text-aligned space, cannot meaningfully manipulate visual tokens (e.g. "replace the dog with a cat").
- **Generative.** The tokens must be decodable back into pixels. Not just semantically — the decoder must preserve *details* the LLM needs to reproduce.

These three properties pull in *different directions*. "Semantic" pushes tokens towards a compact, CLIP-like embedding. "Generative" pushes tokens towards a high-capacity, detail-preserving representation. "Causal" constrains which inter-token dependencies are allowed.

**How SEED-X trains the visual tokeniser.** Two stages (lecture p.37):

*Stage 1 — semantic reconstruction.* Train the Causal Q-Former with an image-text contrastive loss (like CLIP, but with causal constraints), ensuring the tokens are aligned with text meaning. Simultaneously, train the VQ codebook + Reverse Q-Former + (frozen) Stable Diffusion decoder on a reconstruction loss — the decoded image should be semantically close to the original. At this stage, fine details are not expected; the model just needs to reconstruct "a photo of a dog" from the visual tokens, even if the specific dog differs from the input.

*Stage 2 — fine-grained detail recovery.* The same architecture is fine-tuned with a more demanding reconstruction objective, and critically, the Stable Diffusion decoder is now *conditioned on a noisy version of the original image*. This conditional reconstruction forces the reverse path to preserve fine details — if you want to decode the tokens back to the *exact* dog in the input, you need the tokens to carry enough information to do so.

The two-stage training separates "learn to represent meaning" from "learn to recover detail", making optimisation tractable. Without this split, you would ask the tokeniser to handle contrastive alignment and pixel-level reconstruction in one shot — a much harder optimisation landscape.

**Two autoregressive tasks at the LLM level.** Once SEED-X's visual tokens exist, the LLM is trained on two symmetric objectives (lecture p.36):

1. **Image → Text (captioning / VQA)**: Input `[IMG] visual_tokens [/IMG]` followed by a question, predict the text answer.
2. **Text → Image (generation)**: Input text tokens followed by `[IMG]`, predict visual tokens (which will then be decoded to pixels).

Both tasks use the same next-token prediction loss, over the joint vocabulary. The LLM learns to do both image understanding and image generation in one model, with one objective.

**Inference mode — visual feature regression (lecture p.38).** At generation time, when the LLM is asked to produce an image, it outputs a sequence of "learnable queries" in its hidden state — these are $N$ special embeddings that the LLM generates autoregressively alongside any text it is producing. A **visual feature regression loss** (during training) supervises these learnable queries to match the ViT features of the target image. At inference, the learnable queries are fed through the visual de-tokeniser (reverse Q-Former + Stable Diffusion) to produce pixels.

**What to take away.** SEED-X is the cleanest open-source example of an MLLM that *reads* and *writes* images uniformly through one backbone. Every part of the architecture serves the three-way token objective: the causal Q-Former gives you causality, the contrastive training gives you semantic alignment, the two-stage decoder training gives you generative fidelity, and the visual feature regression loss at inference closes the gap between autoregressive logits and the continuous features the decoder wants. When you next read an MLLM paper that claims to do "unified image understanding and generation", mentally check which of these four pieces it solves — most papers only solve two or three.


#### 2.8.3 LLMs as Generator — Visual Scale Autoregressive: VAR *(lecture p.39–40)*

**The core contrast with §2.8.2.** SEED/SEED-X flatten an image into a 1D sequence of patch codes and generate them one-by-one (like GPT reading an image row-by-row). VAR (Visual Autoregressive modeling — Tian et al., ByteDance 2024) does something different: it generates the image at **progressively higher resolutions**, not a flat sequence of patches.

**The motivation.** Natural images have a hierarchical structure: low-frequency information (overall layout, colours) comes before high-frequency detail (edges, textures). 1D next-patch autoregression ignores this — it asks the model to commit to fine details in the top-left of the image before it even knows what the bottom-right looks like. VAR follows the natural coarse-to-fine order.

**Stage 1 — Multi-scale VQ-VAE.** Train a VQ-VAE that encodes each image into a *hierarchy* of discrete token maps $r_1, r_2, \ldots, r_K$, where:
- $r_1$ is a tiny coarse code (e.g. 1×1 or 2×2), representing the global gist.
- $r_2$ is slightly finer (e.g. 3×3).
- ...
- $r_K$ is the full-resolution code (e.g. 16×16 or larger).

Each $r_k$ is a 2D grid of discrete codes at a specific resolution. The total number of codes in the hierarchy is $1^2 + 2^2 + 3^2 + \ldots + 16^2 = \sum_k k^2$, which for $K = 16$ is 1,496 — substantially larger than the 256 codes of a single-scale VQ-VAE.

**Stage 2 — VAR Transformer.** Given the hierarchy, train an autoregressive Transformer to predict each scale given the previous ones:

$$p(r_1, r_2, \ldots, r_K) = \prod_{k=1}^{K} p(r_k \mid r_1, r_2, \ldots, r_{k-1})$$

The Transformer conditions on all previous scales (via a block-wise causal mask where each scale is a "block") and predicts the next scale in a single forward pass. Crucially, the codes *within* a scale are predicted in parallel — only the scales themselves are autoregressive, not the individual patches.

**Why this works.** The Transformer commits to global structure first ($r_1$) and then adds detail ($r_2, \ldots, r_K$). This respects the natural hierarchical structure of images and is also faster at training and inference, because early scales are small and cheap.

**Empirical result.** VAR beat diffusion models on ImageNet FID at the time of release — the first autoregressive image model to do so. It is also much faster to sample from than a 1D autoregressive approach because each scale is predicted in parallel.

**Take-away.** VAR demonstrates that the right inductive bias for image autoregression is *scale*, not *patch*. Expect to see scale-autoregressive ideas in future MLLMs that want to generate high-resolution images without the cost of diffusion sampling.


#### 2.8.4 LLMs as Generator — Visual Diffusion: Transfusion and Show-o

The third flavour of MLLM generation keeps the continuous nature of diffusion (which tends to produce higher-fidelity images than discrete VQ approaches) but wraps it inside a single Transformer that can also handle text.

**Transfusion (Meta, 2024).** One Transformer backbone, two loss heads. On text tokens, the backbone is trained with standard CLM cross-entropy. On image tokens, the backbone is trained with a **diffusion loss** — it predicts the noise that would need to be added to a clean image to produce the current (noisy) image. The same backbone handles both modalities by running different "modality blocks" at different positions in the sequence: text blocks use causal attention, image blocks use bidirectional attention (within an image, because diffusion models do not need causality).

**Show-o.** A similar unified architecture that combines autoregressive text generation with a masked-diffusion-like image generation inside one Transformer. The details of how Show-o handles the text/image boundary are different from Transfusion, but the philosophy is the same: one model, two objectives, trained jointly.

**Why "visual diffusion inside an LLM" is attractive.** Discrete VQ tokens lose fine detail (the VQ quantisation throws information away); diffusion keeps continuous-valued latents and therefore higher visual fidelity. By doing diffusion *inside* the Transformer, you get diffusion-quality images while still having a single model that can also speak text. The price is inference complexity: generating an image now requires multiple diffusion denoising steps for the image block, which is more expensive than emitting discrete tokens.

**Where the field is heading.** As of 2025, visual diffusion inside LLMs is the most promising direction for "one model that does everything" — it combines the flexibility of LLMs (natural-language control, in-context learning) with the visual quality of diffusion models. Expect this to dominate the next generation of MLLMs.


### Part 2 recap

Three ideas to carry into Part 3:

1. **MLLMs are LLMs with a vision encoder and a connector.** Pick one thing from each of §2.4 (vision encoder), §2.5 (connector style), and §1.3 (LLM backbone), and you have specified a VLM.

2. **Connectors come in two styles.** Token-level fusion (LLaVA, Q-Former) turns visual features into tokens and concatenates them with text. Feature-level fusion (Flamingo, CogVLM) injects visual features into the LLM's hidden states via new modules. Token-level won the 2024 open-source landscape for its simplicity.

3. **Non-text output is possible two ways.** Either delegate to a diffusion expert (DiffusionGPT — LLM as conditioner) or give the LLM a visual vocabulary (SEED-X, VAR, Transfusion — LLM as generator). The latter is the more ambitious, and as of 2025, visual diffusion inside a unified LLM backbone (Transfusion) is the most promising variant.

With Part 2 behind us, Part 3 turns to *efficiency*. If we want to train and deploy the 7–70B-parameter VLMs that this section has described, we need LoRA (training-time) and KV Cache (inference-time). Those are the next two big stories.


## Part 3 · LLM Efficiency — LoRA & KV Cache *(lecture p.41–50)*

The lecture groups LoRA and KV Cache into a single chapter titled **"LLM Efficiency = LoRA (Training) + KV Cache (Inference)"** — that is literally the p.41 title slide. This split is the cleanest mental model for "when in the model's lifecycle does each trick fire," and we keep it for Part 3:

- **§3.2 LoRA**. Training-time efficiency. Instead of fine-tuning all 7B parameters of a base model, we train a few million low-rank "delta" parameters that are added on top of the frozen base weights.
- **§3.3 KV Cache**. Inference-time efficiency. During autoregressive decoding, we avoid re-computing the K and V projections for the entire history at every step; we cache them and only project the single new token.

These two optimisations are **completely independent** — you can use one, the other, both, or neither. They are grouped together only because they live under the same "efficiency" umbrella.

### 3.1 Why Efficiency Matters *(lecture p.42)*

Lecture p.42 shows two scatter plots that make the case for efficiency brutally concrete:

- **Training time vs performance.** Horizontal axis: commonsense reasoning score. Vertical axis: training time in millions of GPU-hours. Larger LLaMA variants perform slightly better (72 vs 64 commonsense score), but training cost explodes: LLaMA-2-70B is around 1.7M GPU-hours, LLaMA-2-7B is around 0.2M. That is a roughly 10× cost increase for a few percentage points of benchmark gain.
- **Throughput vs accuracy (with memory).** Horizontal axis: HuggingFace Open LLM Leaderboard score. Vertical axis: throughput in tokens per second. Bubble size: GPU memory used. Larger models are more accurate, slower, and memory-hungry. Moving from a 7B model to a 33B model can halve your throughput *and* double your memory footprint — both at once.

**Conclusion.** Every practical LLM deployment is a constrained-optimisation problem. You have a fixed GPU budget, a latency target, and a quality target, and you have to squeeze the largest / best model into those constraints. LoRA and KV Cache are the two most common levers — one at training time, one at inference. Everyone who has ever deployed an LLM in production has tuned both. Every VLM from Part 2 is trained with LoRA, served with KV Cache, or both. This is not optional knowledge.


### 3.2 Low-Rank Adaptation (LoRA) *(lecture p.43–46)*

LoRA (Hu et al., 2021) is a **parameter-efficient fine-tuning** technique. The question it answers: "how can I adapt a 7B-parameter LLM to a new task without paying the cost of training 7B parameters?" The answer: exploit the empirical observation that the *change* in weights during fine-tuning has low intrinsic rank — train a low-rank *delta* on top of the frozen base weights, and merge the delta back at inference time.

#### 3.2.1 Rank intuition

First, a quick linear-algebra refresher (lecture p.43 makes this point visually).

The **rank** of a matrix is the dimension of the vector space its columns (or rows) span. A $3 \times 3$ matrix has rank at most 3 — three columns can span all of 3D space. A rank-2 matrix has columns that lie in a 2D plane. A rank-1 matrix has columns that all lie along a single line. "Rank" intuitively measures *effective dimensionality* — how many independent directions the matrix can reach.

**The LoRA hypothesis.** When we fine-tune a pre-trained LLM, the change in weights $\Delta W = W_\text{fine-tuned} - W_\text{pre-trained}$ has **low intrinsic rank**. In other words: the model does not need to move in all $d \times d$ independent directions to adapt to a new task; it only needs to move in a few dozen. If we could just learn *those* directions, we would save almost all of the trainable parameters.

Empirically, the hypothesis holds. LoRA at rank $r = 8$ recovers most of the benefit of full fine-tuning on a wide range of tasks, even though rank-8 can capture only $16 \cdot d$ parameters per layer instead of $d^2$. Why? Because fine-tuning is "local" — you are adapting a good model to a small task, not training a new model from scratch. Small local adaptations are, on average, low-rank.

#### 3.2.2 LoRA formulation

Instead of learning $\Delta W$ as a full $d \times d$ matrix, decompose it into two skinny matrices:

$$\Delta W = B A, \quad A \in \mathbb{R}^{r \times d}, \quad B \in \mathbb{R}^{d \times r}$$

The forward pass becomes:

$$h = W_0 x + \Delta W x = W_0 x + B A x$$

where $W_0$ is the frozen pre-trained weight. The LoRA parameters are just $A$ and $B$. Everything else in the model — including $W_0$, all the MLPs, and all the LayerNorms — stays frozen.

**Parameter count.** Full fine-tuning trains $d^2$ parameters per layer. LoRA trains $2 d r$ parameters per layer. For a LLaMA attention projection with $d = 4096$ and $r = 8$:

- Full: $4096^2 = 16{,}777{,}216$ parameters per matrix.
- LoRA: $2 \cdot 4096 \cdot 8 = 65{,}536$ parameters per matrix.
- Ratio: $\frac{65{,}536}{16{,}777{,}216} \approx 0.39\%$.

LoRA is roughly 250× more parameter-efficient than full fine-tuning at $r = 8$, with minimal quality loss.


#### 3.2.3 Why $B = 0$, $A \sim \mathcal{N}(0, \sigma^2)$ initialisation

The initialisation scheme is not arbitrary. There is a hard constraint and a practical preference.

**Hard constraint: at step 0, $\Delta W$ must equal zero.** We are fine-tuning a pre-trained model, so at the start of training the fine-tuned model must match the pre-trained model *exactly*. Otherwise we are starting training far from the initialisation we trust, and the loss at step 0 will be needlessly high. For $\Delta W = B A$ to be zero, at least one of $B$ or $A$ must be zero.

**Subtlety: if both are zero, there is no gradient.** If $B = 0$ and $A = 0$, then both forward and backward passes through the LoRA module are zero, and neither $A$ nor $B$ receives a gradient signal. Training is stuck. So exactly one of them must be non-zero to break the symmetry.

**Practical preference: $B = 0$ and $A \sim \mathcal{N}(0, \sigma^2)$.** With this choice:

- The initial forward pass sees $B A x = 0 \cdot A x = 0$, so the output matches the frozen model exactly.
- The backward pass produces a non-zero gradient for $B$ from step 1 (because $A$ is non-zero and the upstream gradient is non-zero). So $B$ starts moving away from zero.
- The gradient for $A$ is proportional to $B$ and is zero at step 0, but as soon as $B \neq 0$ (step 2 onwards), $A$ also starts receiving gradient.

You could flip the choice ($A = 0$, $B \sim \mathcal{N}$) and it would still work — it just means $A$ is the "waking up" matrix and $B$ is the "always initialised" matrix. The LoRA paper's choice of $B = 0$ is conventional but not unique.

#### 3.2.4 Per-layer application and trainable-parameter accounting

In practice, LoRA is applied to a *subset* of the Transformer layers — typically the attention projections $W_Q$ and $W_V$ of every Transformer block. Experiments in the original paper (Hu et al. 2021) found that applying LoRA to $W_Q$ and $W_V$ is enough; adding $W_K$ helps only marginally, and applying LoRA to the MLP weights or the LayerNorms usually hurts.

**Trainable-parameter accounting for LLaMA-7B with LoRA on $W_Q$ and $W_V$, rank 8:**

- $n_\text{layers} = 32$
- For each layer: two matrices (Q and V), each contributing $2 \cdot d \cdot r = 2 \cdot 4096 \cdot 8 = 65{,}536$ LoRA parameters.
- Per layer: $2 \cdot 65{,}536 = 131{,}072$ trainable parameters.
- Total trainable: $32 \cdot 131{,}072 = 4{,}194{,}304 \approx 4.2$M parameters.
- Full model: ~7B parameters.
- Ratio: $4.2\text{M} / 7\text{B} \approx 0.06\%$.

You are fine-tuning **0.06%** of the model, and in most cases you lose only a fraction of a percent in benchmark performance. This is why LoRA has become the default fine-tuning recipe for open-source LLMs — you can fit a LoRA fine-tune on a single consumer GPU.

**Inference cost.** Zero. At deployment time, you can *merge* the LoRA weights back into the base model: compute $W_0 + BA$ once, store the resulting matrix, and throw away $A$ and $B$. The resulting model has exactly the same parameter count and inference cost as the base model — you pay nothing at runtime for the LoRA fine-tune. This is crucial: LoRA is not a run-time optimisation, it is a *training-time* one.


#### 3.2.5 LoRA variants *(lecture p.46)*

Vanilla LoRA fixes the rank $r$ in advance and applies the same rank to every layer. The variants in the lecture slide try to relax that:

- **DyLoRA (Dynamic LoRA).** At training time, jointly train a *family* of LoRA ranks — $r = 1, 2, 4, 8, 16, 32$ — by randomly sampling a rank at each step and using only the first $r$ rows/columns of $A$ and $B$. At inference time, you can pick any rank you want without retraining. Useful when different deployments have different compute budgets.
- **AdaLoRA (Adaptive LoRA).** Adapt the rank *per layer* during training. Start with a high rank everywhere, measure the importance of each direction (via SVD), and prune unimportant directions. Layers that need more rank get more; layers that do not get their rank reduced. Typically reaches the same final performance as vanilla LoRA with fewer total trainable parameters.
- **IncreLoRA (Incremental LoRA).** Start with a small rank and *grow* it during training as the model needs more capacity. Opposite philosophy to AdaLoRA, similar motivation: do not pay for rank you are not using.

**Bottom line.** All three variants are optimisations over "which layers need how much rank." For Week 9, vanilla LoRA at a fixed $r = 8$ or $r = 16$ is what you need to understand. The variants exist, but they are second-order refinements.

#### 3.2.6 Code — LoRALinear from scratch

The entire LoRA module is about 15 lines. The interesting checks:
1. At initialisation, the LoRA wrapper produces exactly the same output as the frozen base layer.
2. Only $A$ and $B$ are trainable; the base linear's weight is frozen.
3. After a few training steps, $B$ is no longer zero — the LoRA module has started contributing.


In [ ]:
# === Part 3.2.6: LoRALinear from scratch ===
# Demonstrates LoRA's B=0 init and the "zero cost at inference" property.
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, d_in, d_out, rank=8, alpha=16):
        super().__init__()
        # Frozen pretrained weight (would be loaded from a checkpoint in practice).
        self.base = nn.Linear(d_in, d_out, bias=False)
        for p in self.base.parameters():
            p.requires_grad = False

        # Trainable LoRA matrices: A ~ N(0, sigma^2), B = 0.
        self.A = nn.Parameter(torch.randn(rank, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, rank))
        self.scale = alpha / rank

    def forward(self, x):
        # x: (..., d_in)
        base_out = self.base(x)                        # frozen path
        lora_out = (x @ self.A.t()) @ self.B.t()       # trainable path: x -> rank -> d_out
        return base_out + self.scale * lora_out

# --- Check 1: at init, LoRALinear output equals base.Linear output (B=0) ---
torch.manual_seed(0)
d_in, d_out = 64, 32
layer = LoRALinear(d_in, d_out, rank=8, alpha=16)
x = torch.randn(4, d_in)
out_init = layer(x)
out_base = layer.base(x)
max_diff = (out_init - out_base).abs().max().item()
print(f"max |out_init - out_base| at init: {max_diff:.2e}")
assert max_diff < 1e-6, "LoRA at init must equal the base path (B=0)"

# --- Check 2: trainable parameter accounting ---
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
total     = trainable + frozen
print(f"trainable params: {trainable:6d}")
print(f"frozen params:    {frozen:6d}")
print(f"fraction trainable: {100.0 * trainable / total:.2f}%")

# --- Check 3: B moves away from zero after a few gradient steps ---
opt = torch.optim.Adam([p for p in layer.parameters() if p.requires_grad], lr=1e-2)
target = torch.randn(4, d_out)
for _ in range(10):
    loss = (layer(x) - target).pow(2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print(f"B max abs after 10 steps: {layer.B.abs().max().item():.4f}")
assert layer.B.abs().max().item() > 0, "B should now be non-zero"

# --- Check 4: confirm the base weight never moved ---
assert not layer.base.weight.requires_grad
print()
print("LoRALinear: B=0 at init, learns a low-rank delta, base stays frozen.")


### 3.3 KV Cache Deep Dive *(lecture p.47–50)*

LoRA saved us time at training. KV Cache saves us time at **inference**, specifically during the autoregressive decode loop that produces generated tokens one at a time. If you take one thing away from this entire Week 9 material, make it the following: **every production LLM deployment uses KV caching, and no production deployment works without it.** The difference between "with KV cache" and "without KV cache" at 4k-token generation is the difference between *seconds* and *minutes* per response.

This section is long because KV caching has many downstream consequences: it reshapes attention variants (MHA → MQA → GQA), it forces the design of memory-eviction policies, and it is the reason paged attention and vLLM exist.

#### 3.3.1 Step-by-step matrix walkthrough *(lecture p.47–48)*

Let's work through two decode steps by hand to see exactly what gets recomputed and what gets cached.

**Setup.** A decoder-only LLM generating one token at a time. Each decode step consumes the current sequence and predicts the next token. The lecture slides p.47–48 make this concrete with "Step 1" and "Step 2" diagrams.

**Decode Step 1.** We have one token — the start token or a prompt. Call its embedding $x_1$. We compute:

$$Q_1 = x_1 W_Q \quad (1 \times d)$$
$$K_1 = x_1 W_K \quad (1 \times d)$$
$$V_1 = x_1 W_V \quad (1 \times d)$$

Then the attention output at position 1 is:

$$\text{out}_1 = \text{softmax}\left(\frac{Q_1 K_1^\top}{\sqrt{d}}\right) V_1$$

All shapes are $(1, 1)$ and $(1, d)$. The output is a single $d$-dimensional vector, from which the model predicts the next token.

**Decode Step 2 — without caching.** Token 2 arrives ($x_2$). We now need to compute attention at position 2:

$$\text{out}_2 = \text{softmax}\left(\frac{Q_2 \, K_{1:2}^\top}{\sqrt{d}}\right) V_{1:2}$$

The key observation: $Q_2$ only comes from token 2 (it is a single row), but $K_{1:2}$ and $V_{1:2}$ require tokens 1 AND 2. Without a cache, we need to re-project token 1 through $W_K$ and $W_V$ to get $K_1$ and $V_1$ again, even though they are *bit-for-bit identical* to what we computed in Step 1.

**Decode Step 2 — with caching.** Now we cache $K_1, V_1$ from Step 1 in GPU memory. In Step 2, we only need to project the new token $x_2$ once to get $K_2, V_2$, concatenate them with the cached $K_1, V_1$, and run attention:

$$K_{1:2} = \text{concat}(K_1^{\text{cached}}, K_2^{\text{new}})$$
$$V_{1:2} = \text{concat}(V_1^{\text{cached}}, V_2^{\text{new}})$$

We save exactly one re-projection per prior token per decode step. At step $t$, that is a saving of $(t-1)$ projections instead of recomputing the whole history.

**The cumulative saving.** Without caching, producing $T$ tokens requires $1 + 2 + 3 + \ldots + T = \mathcal{O}(T^2)$ total projections. With caching, it requires $T$ projections total. At $T = 1024$, that is a ~500× reduction in projection work. In practice the speedup is smaller (memory bandwidth and attention itself still cost something), but it is easily 5–20× at realistic context lengths.


#### 3.3.2 Why cache K and V but not Q?

This is a common interview question, and the answer has two parts.

**(a) Computational dependency.** At decode step $t$, we compute:

$$\text{out}_t = \text{softmax}\left(\frac{q_t \, K_{1:t}^\top}{\sqrt{d}}\right) V_{1:t}$$

The history $K_{1:t}$ and $V_{1:t}$ will be re-used at *every future step* $t+1, t+2, \ldots$ because each of those steps also needs the full history. So caching them amortises the projection cost across all subsequent steps. In contrast, $q_t$ is used *exactly once*: at step $t$, against the history; at step $t+1$ we need $q_{t+1}$, not $q_t$. There is nothing to amortise.

**(b) Actual length of Q at each decode step.** During the decode phase, we feed exactly **one** new token per step. So $q_t$ is a single row, $q_t \in \mathbb{R}^{1 \times d}$. Only one row of Q is ever needed at a time. Caching previous $q$'s is not only unnecessary — it would *waste* memory.

**The encoder-decoder case makes the argument even stronger.** In a T5-style encoder-decoder, the decoder cross-attends to the encoder's output at every step. The encoder output does not change during decoding — so the K and V for the cross-attention can be computed **once, before decoding starts**, and cached permanently throughout the entire generation. This is why `t5.generate()` and similar encoder-decoder inference APIs are fast despite doing cross-attention: almost all of the cross-attention cost is paid up-front during prefill.

**Take-away.** Q is step-local (used once per step, single row, no reuse). K and V are history-spanning (reused at every future step, grow linearly, amortise perfectly). Cache K and V; do not cache Q.


#### 3.3.3 Memory formula and the LLaMA-7B example *(lecture p.49)*

KV cache is not free — it takes GPU memory, proportional to the sequence length. The memory formula is straightforward:

$$\text{mem} = 2 \cdot n_\text{layers} \cdot \text{batch} \cdot \text{seq\_len} \cdot n_\text{heads} \cdot d_\text{head} \cdot \text{bytes}$$

The leading factor of 2 is because we store **both** K and V — they have the same shape, and each takes the same amount of memory.

**LLaMA-7B worked example.** The config:
- $n_\text{layers} = 32$
- $n_\text{heads} = 32$
- $d_\text{head} = 128$
- dtype: fp16 (2 bytes per number)
- batch size: 4
- seq length: 4096

Plug in:

$$\text{mem} = 2 \cdot 32 \cdot 4 \cdot 4096 \cdot 32 \cdot 128 \cdot 2 \text{ bytes}$$
$$= 8{,}589{,}934{,}592 \text{ bytes}$$
$$\approx 8.0 \text{ GB}$$

**Deployment implication.** On a 24 GB GPU running LLaMA-7B: the model weights occupy ~14 GB (7B params × 2 bytes each), leaving **10 GB** for everything else. The 8 GB KV cache at batch=4 seq=4096 eats almost all of the remaining headroom. If you wanted to serve 8 concurrent requests at the same context length, you would need 16 GB of KV cache, which doesn't fit — you are forced to either shorten the context, reduce the batch size, or use KV-cache optimisations (§3.3.6–3.3.8).

**How to halve the KV cache without changing the model.** Four dials:
1. **Shorter context** (reduce `seq_len`). Linear.
2. **Smaller batch** (reduce `batch`). Linear.
3. **Head sharing via GQA** (reduce effective `n_heads` for K/V). Linear (§3.3.6).
4. **Quantisation** (reduce `bytes`). Linear (§3.3.8).

Every deployed LLM uses some combination of these.

#### 3.3.4 Prefill vs Decode phases

LLM inference has two phases that behave completely differently, and they stress the GPU in opposite ways.

**Prefill phase.** You have the entire prompt — say, 500 tokens. You run one forward pass through the model with all 500 tokens at once. Q, K, V are all $(1, 500, d)$; you compute the attention over the full 500×500 matrix; you populate the KV cache with 500 entries per layer; and you produce a single next-token prediction at the end. Prefill is **compute-bound** — the GPU is doing dense matrix multiplies over long sequences, which is exactly what GPUs are good at.

**Decode phase.** You have the cached K and V from prefill, and a single new token each step. Q is $(1, 1, d)$; K and V are $(1, t, d)$ for the growing history; attention is a $(1 \times t)$ matrix multiplication; you produce one next-token prediction. Decode is **memory-bound** — the GPU is mostly waiting on the KV cache read (the entire $2 \cdot n_\text{layers} \cdot t \cdot d$ chunk of memory must be moved into the compute units), while the actual matrix-multiply is tiny.

**Why this distinction matters.** For prefill, the optimisation priority is "do dense compute efficiently" — Flash Attention, fused kernels, etc. For decode, the optimisation priority is "read less memory per step" — GQA, quantisation, eviction, paged attention. Week 9's KV-cache optimisations are almost entirely about the decode phase, because decode is where chatbot latency lives (prefill is a one-shot cost at the start of a generation; decode fires once per generated token).


#### 3.3.5 Code — Naive vs Cached causal attention + timing

Here is the fully-contained comparison. `NaiveCausalAttention` re-projects the entire history each step (expensive). `CachedCausalAttention` projects only the new token and concatenates with the cache (cheap). Both produce identical outputs (the cache is a pure optimisation, not an approximation) — the only thing that differs is the timing.


In [ ]:
# === Part 3.3.5: Naive vs Cached causal attention + timing ===
# Shows the speedup from caching K, V across decode steps.
import math
import time
import torch
import torch.nn as nn

class NaiveCausalAttention(nn.Module):
    """No cache: at every step, re-project the entire sequence."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.W_o   = nn.Linear(d_model, d_model)

    def forward(self, x_full):
        B, T, D = x_full.shape
        q, k, v = self.W_qkv(x_full).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float("-inf"))
        attn = scores.softmax(dim=-1)
        out  = attn @ v
        return self.W_o(out.transpose(1, 2).reshape(B, T, D))

class CachedCausalAttention(nn.Module):
    """With cache: at every decode step, only project the NEW token."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.W_o   = nn.Linear(d_model, d_model)
        self.cache_k = None
        self.cache_v = None

    def reset(self):
        self.cache_k = None
        self.cache_v = None

    def forward(self, x_new):
        # x_new: (B, 1, D) — only the NEW token
        B, T_new, D = x_new.shape
        q, k, v = self.W_qkv(x_new).chunk(3, dim=-1)
        q = q.view(B, T_new, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T_new, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T_new, self.n_heads, self.d_head).transpose(1, 2)

        if self.cache_k is None:
            self.cache_k, self.cache_v = k, v
        else:
            self.cache_k = torch.cat([self.cache_k, k], dim=2)
            self.cache_v = torch.cat([self.cache_v, v], dim=2)

        scores = q @ self.cache_k.transpose(-2, -1) / math.sqrt(self.d_head)
        attn = scores.softmax(dim=-1)
        out  = attn @ self.cache_v
        return self.W_o(out.transpose(1, 2).reshape(B, T_new, D))

# --- timing harness: simulate a 256-token decode run ---
torch.manual_seed(0)
B, D, H, T = 1, 128, 4, 256
naive  = NaiveCausalAttention(D, H).eval()
cached = CachedCausalAttention(D, H).eval()
cached.W_qkv.load_state_dict(naive.W_qkv.state_dict())
cached.W_o.load_state_dict(naive.W_o.state_dict())
tokens = torch.randn(B, T, D)

# Naive: re-run on [0:t+1] each step (simulating "no cache")
t0 = time.perf_counter()
with torch.no_grad():
    for t in range(1, T + 1):
        _ = naive(tokens[:, :t, :])
naive_time = time.perf_counter() - t0

# Cached: feed one token at a time, reusing the cache
cached.reset()
t0 = time.perf_counter()
with torch.no_grad():
    for t in range(T):
        _ = cached(tokens[:, t:t+1, :])
cached_time = time.perf_counter() - t0

print(f"Naive decode:  {naive_time*1000:7.1f} ms")
print(f"Cached decode: {cached_time*1000:7.1f} ms")
print(f"Speedup:       x{naive_time/cached_time:.1f}")
assert cached_time < naive_time, "Cached should be faster"
print()
print("KV cache removes redundant recomputation of history.")


#### 3.3.6 Attention head variants: MHA → MQA → GQA *(lecture p.49)*

The KV cache memory formula is proportional to $n_\text{heads}$. If we could reduce the number of *K/V* heads without reducing the number of *Q* heads, we would shrink the cache proportionally. That is exactly what Multi-Query Attention and Grouped-Query Attention do.

**MHA (Multi-Head Attention).** Standard multi-head attention. If the model has $H$ heads, then Q has $H$ heads, K has $H$ heads, V has $H$ heads. Each head has its own independent $W_Q^{(h)}, W_K^{(h)}, W_V^{(h)}$ projection. The KV cache stores $2 \cdot H$ head slices per layer per token. This is what we implemented in §3.3.5.

**MQA (Multi-Query Attention, Shazeer 2019).** All $H$ query heads share ONE key head and ONE value head. The projection matrices become $W_Q$ producing $H \cdot d_\text{head}$ outputs but $W_K$ and $W_V$ producing only $1 \cdot d_\text{head}$ outputs. At attention-time, the single K head is broadcast across all $H$ Q heads. KV cache shrinks by a factor of $H$ — a huge saving. Quality drops noticeably, though, because there is less head-level diversity in the K/V representations.

**GQA (Grouped-Query Attention, Ainslie et al. 2023).** The middle ground. Split the $H$ Q heads into $G$ groups, where $G < H$ but $G > 1$. Each group shares one K head and one V head. So there are $H$ Q heads but only $G$ K heads and $G$ V heads. The KV cache shrinks by a factor of $H / G$. At $H = 32$, $G = 8$ (LLaMA-2 70B's configuration), the cache is 4× smaller than MHA with minimal quality loss.

**Why GQA won.** MHA is expensive, MQA hurts quality, GQA gives you a 4× KV-cache reduction at essentially no quality loss. LLaMA-2 70B, Mistral 7B, Qwen, Gemma, and almost every open-source 2024 LLM use GQA. This one architectural change has probably saved more GPU-hours than any other LLM optimisation of the past two years.


In [ ]:
# === Part 3.3.7: Grouped-Query Attention from scratch ===
# n_heads Q heads share n_kv_heads K/V heads. KV cache shrinks proportionally.
import math
import torch
import torch.nn as nn

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_head     = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.W_q = nn.Linear(d_model, n_heads    * self.d_head)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_head)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_head)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        q = self.W_q(x).view(B, T, self.n_heads,    self.d_head).transpose(1, 2)  # (B, Hq, T, d)
        k = self.W_k(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)  # (B, Hkv, T, d)
        v = self.W_v(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        # Expand K, V so each group of Q heads shares the same K/V
        k = k.repeat_interleave(self.group_size, dim=1)  # (B, Hq, T, d)
        v = v.repeat_interleave(self.group_size, dim=1)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        attn = scores.softmax(dim=-1)
        out  = attn @ v
        return self.W_o(out.transpose(1, 2).reshape(B, T, D))

# --- KV cache size accounting ---
torch.manual_seed(0)
d_model = 4096
n_heads = 32
print(f"{'variant':10s}  {'n_kv_heads':12s}  {'KV proj params':16s}  cache factor")
for n_kv_heads in [32, 8, 4, 1]:
    layer = GroupedQueryAttention(d_model, n_heads, n_kv_heads)
    kv_params = sum(p.numel() for n, p in layer.named_parameters() if n.startswith(("W_k", "W_v")))
    label = {32: "MHA", 1: "MQA"}.get(n_kv_heads, f"GQA({n_kv_heads})")
    print(f"{label:10s}  n_kv_heads={n_kv_heads:<3d}  {kv_params:>12d}      x{n_kv_heads/n_heads:.3f}")

# --- smoke test: forward pass for a few GQA variants ---
x = torch.randn(2, 10, d_model)
for n_kv in [32, 8, 1]:
    layer = GroupedQueryAttention(d_model, n_heads, n_kv)
    out = layer(x)
    assert out.shape == (2, 10, d_model)
print()
print("GQA with n_kv_heads in {32, 8, 1} all produce the same output shape.")


#### 3.3.8 KV-cache quantisation

The memory formula has one more dial we have not touched: `bytes`. The KV cache is normally stored in fp16 (2 bytes per element). Quantising it to int8 halves the memory footprint, int4 quarters it.

**Why KV quantisation works well.** The K and V tensors are mostly used for inner products (Q·K) and weighted sums (attn·V). Both operations tolerate quantisation noise fairly well compared to, say, weight quantisation, because the noise in K and V tends to *average out* across many tokens and heads. Production systems (vLLM, TensorRT-LLM) ship with int8 and fp8 KV cache support, and the quality impact is usually <1% on standard benchmarks.

**Concrete impact on the LLaMA-7B example.** From §3.3.3, fp16 KV cache at batch=4, seq=4096 was 8.0 GB. Switching to int8:

$$\text{mem}_\text{int8} = 8.0 \text{ GB} \times \frac{1 \text{ byte}}{2 \text{ bytes}} = 4.0 \text{ GB}$$

Suddenly we have 10 GB of headroom for KV cache instead of 2 GB — enough for 10 concurrent requests instead of 5. int8 KV cache alone roughly doubles the serving throughput on memory-bound deployments.

**Combined with GQA.** Stacking KV quantisation on top of GQA(n_kv_heads=8): fp16 MHA KV cache was 8 GB; with GQA it becomes 2 GB; with GQA + int8 it becomes 1 GB. That is an 8× reduction, and it is how a 24 GB GPU can serve 40+ concurrent LLaMA-2-70B requests at 4k context.

#### 3.3.9 Eviction policies *(lecture p.50)*

When the context gets truly long — 100k+ tokens — even GQA + int8 quantisation is not enough. At some point you have to *evict* tokens from the cache, keeping only a subset. Lecture p.50 shows four families:

- **(a) Dense Attention.** Keep every token. The default, and the only policy that is guaranteed to match the un-evicted model output. Costs $\mathcal{O}(T)$ cache memory. For contexts up to ~32k tokens, this is usually what you want.
- **(b) Window Attention.** Keep only the most recent $W$ tokens, where $W$ is a fixed window (say, 1024). Evict older tokens. Deterministic, constant memory, but loses information from the beginning of the context.
- **(c) Initial Token / Attention Sink.** Surprising empirical discovery (Xiao et al. 2024): LLMs learn to dump excess attention weight onto the first few tokens of a sequence — even if those tokens carry no semantic information. Evicting those tokens causes the model's output to *catastrophically degrade*, because the attention distribution has no "sink" to absorb its leftover probability mass. Fix: always keep the first 4 tokens (the "attention sink") plus a recent window. Very effective and still constant memory.
- **(d) Dynamic / Attention-based eviction.** At each step, score tokens by their attention weight and evict the low-scoring ones. Adaptive but expensive to compute; usually reserved for extreme long-context scenarios where other policies have been exhausted.

**Take-away.** If you need to serve 100k+ token contexts on fixed memory, *some* eviction policy is non-negotiable, and attention-sink-aware window is almost always the right first thing to try.

#### 3.3.10 Paged Attention (conceptual only) *(lecture p.49)*

**The problem.** Naive KV cache allocation assigns a contiguous block of memory per request — large enough to hold the maximum sequence length you might generate. If you serve 100 concurrent requests with a 4096-token max, you pre-allocate 100 × 4096 worth of KV cache even if most requests only generate 200 tokens. This causes massive memory fragmentation and wasted space.

**The solution — borrow from OS virtual memory.** Split the KV cache into fixed-size *pages* (blocks of, say, 16 tokens). Each request maintains a *page table* that points at the pages holding its KV entries; those pages can be non-contiguous in physical memory. When a request grows by another 16 tokens, it allocates one more page; when a request finishes, its pages are freed and returned to the pool.

**Why this matters.** Paged attention eliminates KV fragmentation, which in practice doubles or triples the effective batch size you can serve on a given GPU. It is the core idea behind **vLLM**, the most widely-deployed open-source LLM inference server as of 2025.

**We do not implement paged attention here.** It is a systems-level optimisation that lives in the memory allocator and kernel scheduler, not in the model architecture. The right thing to do if you need it is use vLLM or TensorRT-LLM; the right thing to know for this course is "it exists, and it is why vLLM is so much faster than naive HuggingFace inference."


## Part 4 · Mixture of Experts (MoE) *(lecture p.51–56)*

Parts 3 taught us two efficiency tricks: LoRA trains fewer parameters, KV Cache skips redundant inference work. Part 4 introduces a third, architectural, efficiency trick: **Mixture of Experts**, which lets a model have enormous total parameter count while paying only a fraction of the per-token compute. If LoRA is "train less" and KV Cache is "compute less," MoE is "activate less."

### 4.1 Motivation — parameters vs inference time *(lecture p.52)*

Lecture p.52 shows a benchmark table from the LLaMA-1 and LLaMA-2 papers. Two empirical assumptions fall out of it:

- **More parameters ⇒ higher performance.** LLaMA-2 70B scores 71.9 on commonsense reasoning, LLaMA-2 7B scores 63.9. The gap is real and consistent across benchmarks.
- **More parameters ⇒ larger inference time.** 70B requires roughly 10× more FLOPs per token than 7B, and the memory footprint grows proportionally.

These two assumptions pull in opposite directions, and the naive response — "just pick the biggest model that fits" — is unsatisfying. Is there a way to get the *quality* of a 70B model while paying the *inference cost* of a 7B model?

**Mixture of Experts says yes, under one condition: different tokens need different expertise.** If every token needed the full 70B of parameters, there would be no way to cheat. But in reality, most tokens only need a *sliver* of the model's capacity — a token inside a Python function call needs the "code expert"; a token in a Chinese sentence needs the "Chinese expert"; a token in a mathematical derivation needs the "math expert". If we can learn to route each token to the right sliver, we can have a 70B-parameter *model* that runs like a 7B-parameter *compute graph* at each step.

The cost: you still pay the 70B memory footprint (all experts must be in memory for routing), but you pay only the 7B compute. For inference on a single GPU, this trade is usually a win — GPU memory is plentiful, GPU FLOPs are scarce.


### 4.2 MoE Layer Structure and Gating Network *(lecture p.53)*

Lecture p.53 diagrams an MoE layer as a drop-in replacement for the MLP / FFN in a standard Transformer block. Rather than "one big MLP", the layer contains $n$ **experts** (each is its own small MLP) and a **gating network** that decides which experts each token should go through.

**The layer structure:**

1. **Input token** $x \in \mathbb{R}^d$.
2. **Router (gating network)** computes logits $g(x) \in \mathbb{R}^n$ and turns them into routing weights.
3. **Expert selection** picks the top-$k$ experts (typically $k = 1$ or $k = 2$) and discards the rest.
4. **Each selected expert** runs $E_i(x)$ — a standard MLP with its own independent parameters.
5. **Weighted sum** of the selected experts' outputs: $y = \sum_{i \in \text{top-}k} g_i(x) \cdot E_i(x)$.

The rest of the Transformer block — self-attention, LayerNorm, residual — is unchanged. MoE only replaces the MLP/FFN.

**The key observation.** If $k = 2$ and $n = 8$, then at any given forward pass we activate $2/8 = 25\%$ of the expert parameters per token. The model has 8× the total parameters of a non-MoE model, but the per-token compute is only 2× (because we run 2 experts instead of 1 MLP). Net effect: an 8-expert MoE layer with top-2 routing has roughly 4× the effective capacity-per-compute of a standard MLP.

**Which Transformer blocks use MoE?** In practice, MoE is applied to a *subset* of blocks — alternating blocks in most architectures (Mixtral, DeepSeek-MoE, GLaM), every block in some (Switch Transformer). Whether to MoE-ify every block or only some is a hyperparameter that trades model capacity against router load imbalance.


### 4.3 Sparse Gating Formula *(lecture p.54)*

Lecture p.54 lists the sparse gating formula in three steps. This is what the gating network actually computes.

**Step 1 — Noise injection.** The router computes a noisy version of its logits:

$$H(x)_i = (x \cdot W_g)_i + \mathcal{N}(0, 1) \cdot \text{softplus}\!\left((x \cdot W_\text{noise})_i\right)$$

Two learned matrices: $W_g$ (the "clean" routing weights) and $W_\text{noise}$ (a per-expert noise scaling). For each expert $i$, the logit is the clean dot product plus Gaussian noise whose magnitude is controlled by a learned softplus of another dot product. The noise is only added during training; at inference we drop it.

**Why add noise at all?** Without noise, the top-$k$ selection is a deterministic function of the input. If two experts have very close logits, the router *always* picks the same one, and the "loser" expert never receives any training signal. Noise breaks the tie stochastically, so that both experts get a chance to be updated. This is load balancing by randomisation.

**Step 2 — Top-$k$ mask.** Define:

$$\text{KeepTopK}(v, k)_i = \begin{cases} v_i & \text{if } v_i \text{ is among the top } k \text{ elements of } v \\ -\infty & \text{otherwise} \end{cases}$$

Everything not in the top $k$ gets set to $-\infty$. This is how we enforce sparsity — only the top $k$ experts will have a non-zero gate after softmax.

**Step 3 — Softmax.** Normalise the masked logits into routing weights:

$$G(x) = \text{softmax}(\text{KeepTopK}(H(x), k))$$

Because the non-selected experts have logit $-\infty$, their softmax output is $0$. The selected $k$ experts have positive gates that sum to 1. The final MoE output is $y = \sum_i G(x)_i \cdot E_i(x)$ — a weighted sum over the top $k$ experts.

**Putting it together.** The router is a $d \to n$ linear layer with a noise term and a top-$k$ softmax. It is tiny compared to the experts themselves, but it does all the "work" of deciding which expert handles which token. Get the router right and you unlock the efficiency advantage of MoE; get it wrong and the routing collapses and the advantage vanishes.


### 4.4 From-scratch — TopK gating + MoE FFN layer

The code below implements the three-step gating formula above, plus a minimal MoE FFN layer that dispatches tokens to the selected experts and computes the gated sum. Two checks:

1. After top-$k$ masking, each token has exactly $k$ non-zero gates (sparsity check).
2. The output shape matches the input shape (this is a drop-in replacement for an MLP).

Note: the dispatch pattern in this code is *unoptimised* — it runs every expert on every token and then multiplies by the gate to zero out the non-selected ones. Production implementations use per-expert batched dispatch to actually save compute. The pedagogical simplicity is worth the performance cost at this scale.


In [ ]:
# === Part 4.4: TopK noisy gating + MoE FFN ===
import torch
import torch.nn as nn
import torch.nn.functional as F

class TopKGating(nn.Module):
    """Shazeer et al. (2017) sparsely-gated MoE router."""
    def __init__(self, d_model, n_experts, k=2):
        super().__init__()
        self.k = k
        self.n_experts = n_experts
        self.W_g     = nn.Linear(d_model, n_experts, bias=False)
        self.W_noise = nn.Linear(d_model, n_experts, bias=False)

    def forward(self, x):
        # x: (B, T, D) -> gates: (B, T, n_experts), sparse
        clean_logits = self.W_g(x)
        if self.training:
            noise_scale = F.softplus(self.W_noise(x))
            noise = torch.randn_like(clean_logits) * noise_scale
            logits = clean_logits + noise
        else:
            logits = clean_logits

        topk_vals, topk_idx = logits.topk(self.k, dim=-1)          # (B, T, k)
        mask = torch.full_like(logits, float("-inf"))
        mask.scatter_(-1, topk_idx, topk_vals)
        gates = F.softmax(mask, dim=-1)                             # (B, T, n_experts)
        return gates, topk_idx


class MoEFFN(nn.Module):
    """Drop-in replacement for a standard Transformer FFN."""
    def __init__(self, d_model, d_ff, n_experts, k=2):
        super().__init__()
        self.gating = TopKGating(d_model, n_experts, k)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
            for _ in range(n_experts)
        ])

    def forward(self, x):
        # x: (B, T, D). Unoptimised dispatch: run all experts, take the gated sum.
        gates, _ = self.gating(x)                                  # (B, T, n_experts)
        expert_outs = torch.stack([e(x) for e in self.experts], dim=-2)  # (B, T, n_experts, D)
        out = (gates.unsqueeze(-1) * expert_outs).sum(dim=-2)
        return out


# --- sparsity check: each token should have exactly k non-zero gates ---
torch.manual_seed(0)
B, T, D, D_FF, N, K = 2, 5, 32, 64, 4, 2
moe = MoEFFN(D, D_FF, n_experts=N, k=K).train()
x = torch.randn(B, T, D)
out = moe(x)

gates, idx = moe.gating(x)
nonzero_per_token = (gates > 0).sum(dim=-1)
unique = nonzero_per_token.unique().tolist()
print(f"nonzero gates per token (unique values): {unique}  (expect [{K}])")
print(f"output shape: {tuple(out.shape)}   (expect (B, T, D) = ({B}, {T}, {D}))")
assert out.shape == (B, T, D)
assert (nonzero_per_token == K).all(), "Exactly k experts should be active per token"

# --- inference-mode check: no noise added, routing should be deterministic ---
moe.eval()
gates_a, idx_a = moe.gating(x)
gates_b, idx_b = moe.gating(x)
assert torch.equal(idx_a, idx_b), "At eval-time, routing must be deterministic"
print()
print("TopK MoE: each token routes through exactly k experts; eval-time routing is deterministic.")


### 4.5 Load-balance loss — why TopK alone is not enough

The TopK gating formula above has a subtle failure mode: if one expert starts out slightly better than the others, it will be picked more often, get more training signal, become even better, get picked even more often, and eventually monopolise *all* tokens. The other experts get no gradient signal, stay random, and are effectively dead weights. This is **routing collapse**, and it is the single biggest practical issue in MoE training.

**The fix: an auxiliary load-balance loss.** The Switch Transformer paper (Fedus et al. 2021) proposes a canonical form. For each expert $i$:

- $f_i$ = the fraction of tokens in the current batch that were routed to expert $i$ (how *often* expert $i$ was picked).
- $P_i$ = the mean gate value for expert $i$ across the batch (how *strongly* expert $i$ was picked when it was picked).

The auxiliary loss is:

$$\mathcal{L}_\text{aux} = n \cdot \sum_{i=1}^{n} f_i \cdot P_i$$

Intuitively: penalise experts that are *both* popular (high $f_i$) AND highly weighted (high $P_i$). The only way to minimise this product is to spread the load evenly — if one expert dominates, its $f_i \cdot P_i$ shoots up; if load is uniform, the sum is minimised.

The total training loss becomes $\mathcal{L} = \mathcal{L}_\text{LM} + \lambda \cdot \mathcal{L}_\text{aux}$, where $\lambda \approx 0.01$ in practice.

**Why this is not in the lecture slide but is in this notebook.** The slide shows the sparse gating formula but not the balance loss. Without the balance loss, a from-scratch MoE like the one above would collapse to one expert within a few hundred training steps. Anyone who tries to reproduce MoE training from the slide alone will hit this failure mode immediately. The balance loss is not optional in practice — it is the difference between "MoE works" and "MoE collapses."


### 4.6 DS-MoE — Dense training, sparse inference *(lecture p.55)*

Sparse top-$k$ routing has one more training-time problem: the discrete argmax in `KeepTopK` produces *zero gradient* for the non-selected experts. Training can only update the expert that was picked, not the ones that *almost* won. This makes learning the router slow and unstable.

**DS-MoE (Pan et al. 2024) — dense train, sparse infer.** At training time, run a **dense** forward pass where every expert sees every token:

- Every expert produces an output for every token.
- The router produces weights for all experts (no top-$k$ mask).
- The output is the full gated sum $\sum_i G_i(x) \cdot E_i(x)$ over *all* experts.
- An auxiliary mutual-information loss encourages the router's weights to be sparse anyway, so the router learns to put most of its weight on a few experts.

At inference, fall back to standard sparse top-$k$ routing — only the top-$k$ experts are activated, and the rest are skipped. This gives the best of both worlds: rich gradient signal at training time (dense), cheap inference at deployment (sparse).

**Trade-off.** Dense training is *much* more expensive than sparse training (you pay $n \times$ the expert compute per step), but the resulting model is more stable and often reaches higher final quality. For big-budget pre-training runs (Switch Transformer scale), DS-MoE is worth the extra training compute. For smaller-budget runs, plain sparse training with balance loss is usually fine.


### 4.7 MoE-LLaVA — Mixture of Experts for Vision-Language Models *(lecture p.56)*

MoE is not just for text LLMs. The lecture closes the MoE chapter with MoE-LLaVA (Lin et al. 2024), which applies MoE to a vision-language model. The interesting thing is the *training pipeline* — you cannot just plug MoE into a pretrained LLaVA and expect it to work.

**MoE-LLaVA's three-stage training recipe:**

1. **Stage 1 — Base VLM pre-training.** Train the projection MLP from scratch on caption data, matching LLaVA's Stage 1 (§2.7.2). The LLM backbone and vision encoder are frozen. No MoE yet.

2. **Stage 2 — Instruction tuning.** Unfreeze the LLM backbone and train on visual instruction data, matching LLaVA's Stage 2. Still no MoE. At the end of this stage, we have a standard (non-MoE) LLaVA.

3. **Stage 3 — MoE-ify the FFN layers.** For each FFN in the LLM backbone, *copy* its weights $N$ times to create $N$ identical expert initialisations. Add a newly-initialised router on top. Train only the router and the experts (the vision encoder, projector, and attention layers stay frozen). Each expert starts from the same weights, but because the router sends different tokens to different experts, the experts diverge during training and specialise.

**Why this matters.** Stage 3 is MoE "bootstrapping from a dense model" — you do not train the experts from scratch. That saves a huge amount of compute, and it also provides a good initialisation for the experts (they start from a known-good FFN). The final MoE-LLaVA has ~3× the parameter count of the dense LLaVA it was built from, similar per-token inference cost (top-$k$ with $k$ small), and better quality on visual reasoning benchmarks.

**Take-away.** MoE-LLaVA is the archetype for "how to MoE-ify any pretrained Transformer." The same recipe — Stage 3 expert-copying — can be applied to LLaMA, Qwen, or any other dense LLM. Expect to see this technique used in more open-source MLLMs throughout 2025.


## Part 5 · Multi-modal Foundation Models in Robotics *(lecture p.57–66)*

### 5.1 Motivation — from perception to action

Up to this point, "multi-modal" has meant *reading* non-text inputs. A VLM that can say "this is a cup on a table" has solved a vision-language task. But a VLM that can say that does not, by itself, *pick up* the cup. Robotics requires a mapping from (language instruction, visual observation) to (physical action) — a mapping that breaks the classical VLM mold, because the output modality is neither text nor pixels but a sequence of motor commands.

Lecture p.57–66 presents two ways to bridge this gap:

- **VLMs for Perception (§5.2).** Keep the VLM frozen and use it as a reasoning module that produces *structured intermediate outputs* — points, constraints, code — which a classical planner turns into robot actions. The VLM never directly controls the robot; it is a high-level decision-maker sitting on top of a traditional stack.
- **VLAs — Vision-Language-Action models (§5.3–5.8).** End-to-end. Extend the VLM's output vocabulary to include *motor-action tokens*, and let the model generate actions the same way it generates text — next-token prediction over an enlarged vocabulary. The model *is* the controller.

Both approaches are active research directions as of 2025. The "VLM for perception" approach has the advantage of reliability (you keep a battle-tested motion planner in the loop); the "VLA" approach has the advantage of generality (one model handles everything). We look at two papers from each category.


### 5.2 VLMs for Robotic Perception

#### 5.2.1 RoboPoint — VLM for Action Points *(lecture p.58)*

**The task.** Given an RGB image and an instruction like "find a few points in the vacant area to the left of the object marked by the red box," the VLM must output 2D pixel coordinates that a robot motion planner can turn into reach targets. The VLM never touches the robot's joints directly — it just emits coordinates as text.

**Architecture (lecture p.58, middle panel).** Standard LLaVA-style VLM: frozen image encoder → learnable MLP projector → Vicuna-1.5 13B (fine-tuned). The output format is *literally a text string containing coordinate tuples* like `[(0.54, 0.23), (0.39, 0.45), (0.48, 0.60)]`. The LLM emits these coordinates as ordinary token sequences; a downstream parser extracts the floats and converts them to pixel positions.

**Training data — procedurally generated.** This is the most interesting part of RoboPoint. Training a VLM to output coordinates requires labelled (image, instruction, coordinate) triples, and there is no such dataset at scale. RoboPoint *procedurally generates* its own:

1. **Scene generation.** Randomly lay out 3D objects in a simulator — boxes, shelves, drawers, clutter.
2. **Rendering.** Render RGB images from multiple camera angles.
3. **Auto-labelling.** Use the known 3D geometry to automatically generate spatial relations ("behind obj1, obj5", "left of obj2, obj3", "between obj3, obj6, obj8") and corresponding pixel coordinates.

This produces millions of (image, instruction, coordinate) triples for zero human labelling cost. The VLM is then instruction-tuned on this synthetic data (plus a small amount of real-world data for domain transfer).

**Real-world execution (lecture p.58, bottom panel).** Given a real image from a robot camera and a natural-language instruction, RoboPoint emits 2D action points. The system projects those points into 3D using a depth camera, and a classical motion planner drives the gripper to the target. The VLM's role is specifically "convert fuzzy spatial language into precise pixel coordinates"; everything downstream of the coordinates is standard robotics.

**Why this matters.** It shows that VLMs can output *structured* data (points, boxes, masks, code) by treating the structured data as text — no architectural changes required. The same trick generalises to any structured output the VLM can tokenise.


#### 5.2.2 ReKep — VLM for Keypoint Constraints *(lecture p.59–60)*

ReKep (Huang et al., CoRL 2024) takes the "VLM as reasoning engine" philosophy one level higher. Instead of outputting coordinates, ReKep outputs *constraints* between keypoints, and a classical constrained-optimisation solver turns those constraints into smooth robot trajectories.

**The pipeline (lecture p.59).**

1. **Large Vision Model proposes keypoints.** A general-purpose vision model like SAM or DinoV2 proposes semantically meaningful keypoints on the objects in the scene — the rim of a teapot, the rim of a cup, the handle of the teapot, etc.
2. **VLM reads the scene and the instruction, generates constraint code.** The user says "pour tea into the cup." The VLM (GPT-4V or Gemini) looks at the image plus the keypoints and generates Python code describing the *constraints* the robot must satisfy:
   ```python
   def subgoal_stage1_f1(k):
       dist = norm(k[0] - k[1])
       return dist  # teapot rim distance to cup rim — should be small

   def path_stage2_f1(k):
       z_diff = abs(k[1] - k[2])
       return z_diff  # teapot spout should stay above cup rim

   def subgoal_stage2_f1(k):
       k[3][2] += 0.10  # teapot should tilt to pour
       return norm(k[2] - k[3])
   ```
3. **Constrained optimisation solver.** Given the symbolic constraints as Python functions, a solver (e.g. IPOPT) computes a robot trajectory that minimises the constraint violations over time. The solver is a classical piece of robotics machinery — nothing VLM-related.

**Why the constraints approach is powerful.** Constraints *generalise* — the same "pour tea into cup" constraints describe pouring coffee, pouring water, or pouring soup, as long as the keypoints are semantically labelled. Instructions like "fold the t-shirt in half" and "fold the linen into thirds" are also expressible as keypoint constraints that work across a wide range of objects and scenes.

**Lecture p.60 concrete example.** Folding a rainbow-striped t-shirt with a dual-arm robot. The VLM identifies keypoints on the t-shirt (sleeves, collar, hem corners), writes constraints ("sleeves should touch each other", "hem should meet collar"), and the solver produces a two-arm coordination trajectory that actually folds the shirt. Neither the VLM nor the solver has ever seen a rainbow t-shirt before — the VLM's reasoning is generic ("a t-shirt has these keypoints"), and the solver's optimisation is generic ("minimise these distances"). The combination generalises.

**Take-away.** ReKep shows that VLMs can be *reasoning engines* for robotics without ever producing raw motor commands. The VLM writes *code*; classical robotics machinery executes the code. This is arguably the most reliable deployment recipe for VLMs in robotics circa 2024–2025, because the human-auditable intermediate (constraint code) can be inspected, tested, and safely constrained.


### 5.3 From VLM to VLA — PaliGemma *(lecture p.61)*

The alternative to "VLM as reasoning engine" is to have the VLM produce actions *directly*. PaliGemma (Google DeepMind, 2024) is the archetype for this approach and is the base model used by many subsequent VLAs.

**Architecture (lecture p.61).** SigLIP 400M vision encoder → linear projection → Gemma 2B language decoder. The connector is a single linear layer — even simpler than LLaVA's two-layer MLP.

**Why SigLIP instead of CLIP?** SigLIP is a CLIP variant that replaces the softmax contrastive loss with a sigmoid loss (Zhai et al. 2023). It trains more efficiently at large batch sizes and ends up producing visual features with slightly better per-pixel resolution, which matters for robotics where you often care about spatial details.

**Why Gemma 2B instead of LLaMA 7B?** Size. PaliGemma is specifically designed to run at robot control rates (10–20 Hz), which means the entire forward pass — vision encoder + connector + LLM — must complete in 50–100 ms. LLaMA 7B is too slow for that budget on most deployment hardware. Gemma 2B fits the latency window with room to spare.

**What PaliGemma is used for.** Out of the box, PaliGemma is a general-purpose VLM — it can answer "where is the photographer resting?" type questions about an image. But its real role is as a **small, fast, pretrained base model** for downstream VLAs. OpenVLA (§5.4), Physical Intelligence's π₀ and π₀.₅ (§5.7), and other 2024–2025 robotics models all use either PaliGemma or a similarly small VLM as their starting point.

**The underlying lesson.** Robotics cannot afford the latency of a 70B-parameter model. Every VLA from 2024 onwards is built on a base model ≤13B parameters, and most are in the 2–7B range. This constraint drives the architecture choices.


### 5.4 VLA Architecture — OpenVLA *(lecture p.62)*

OpenVLA (Kim et al., CoRL 2024 — Stanford + Google) is the most influential open-source VLA to date. It is the closest thing the field has to an "LLaVA for robots": a simple, reproducible recipe that can be retrained by anyone with a modest GPU budget.

**Architecture (lecture p.62 figure).**

1. **Vision encoder — DinoV2 + SigLIP, concatenated.** Two vision encoders run in parallel on the same input image, and their features are concatenated. DinoV2 contributes self-supervised spatial features (good at "where is the object"); SigLIP contributes contrastively-aligned features (good at "what is the object"). The combination is more informative than either alone.
2. **Connector — MLP projector.** Standard LLaVA-style: a 2-layer MLP maps the concatenated visual features into the LLM's token embedding space.
3. **LLM backbone — LLaMA-2 7B.** The language model (fine-tuned, not LoRA — OpenVLA does full fine-tuning). The LLaMA-2 tokenizer is re-used, with a few modifications described below.
4. **Action de-tokenizer.** The single most important design choice: OpenVLA does not add a separate "action head" on top of LLaMA. Instead, it *extends the LLaMA vocabulary* to include action tokens.

**How action tokenisation works.** A 7-DoF robot action — say, `(Δx, Δy, Δz, Δroll, Δpitch, Δyaw, ΔGrip)` — is first discretised. Each continuous action component is binned into 256 values. OpenVLA then takes the *last 256 tokens* of the LLaMA-2 tokenizer (which were originally rare text tokens) and *reassigns* them to the action bins. So "LLaMA-2 token 31999" now means "gripper position 255" instead of whatever rare word it originally meant. During fine-tuning, the LLM learns to emit these reassigned tokens in response to "what should the robot do next" prompts, and a simple detokenizer converts the emitted tokens back to continuous action values.

**Why reassign existing tokens instead of adding new ones?** Adding new tokens would require expanding the embedding table and the output projection, which is fiddly and introduces uninitialised parameters. Reassigning existing tokens means you only need to fine-tune — no architecture surgery. The rare text tokens being overwritten are, by definition, rarely used, so you lose almost no text generation capability.

**Input format.** A typical OpenVLA training sample looks like:

```
Image: [visual tokens from DinoV2+SigLIP through the MLP projector]
Instruction: "Put eggplant in bowl"
Prompt: "What should the robot do to {task}? A:"
Response: [action token 1] [action token 2] ... [action token 7]
```

The LLM sees the visual tokens followed by the instruction and a prompt, and is trained (with next-token prediction) to emit the 7 action tokens. At inference, you sample 7 tokens, decode them to a continuous 7-DoF action, and send that action to the robot.

**This is the cleanest possible recipe for multi-modal output by vocabulary extension.** Exactly the SEED-X pattern from §2.8.2, but with robot actions instead of pixels. If you understand OpenVLA, you understand how modern VLAs work.


### 5.5 VLA Training Paradigm *(lecture p.63)*

OpenVLA's training paradigm is deliberately simple and mirrors standard LLM practice.

**Two-phase training.**

*Phase 1 — Post-training on large robot data.* Start from the pretrained (text) LLaMA-2 7B + DinoV2 + SigLIP + MLP projector. Collect a large, diverse robot dataset — Open X-Embodiment, which aggregates 970k robot episodes across 22 robot embodiments, 500+ tasks, and dozens of labs. Train the full model (LLM + projector + fine-tune) with the action-token next-prediction loss on this dataset. This takes **14 days on 64 A100 GPUs**. The result is a general-purpose robot policy that has seen a vast diversity of manipulation tasks and can handle many of them in zero-shot.

*Phase 2 — Fine-tuning on specific task.* Take the post-trained model from Phase 1 and fine-tune it on a small dataset for your specific deployment — your robot, your gripper, your task. This might be a few hundred episodes of "pick up banana", "pour water", etc. Training takes **1 day on 1 GPU** because the dataset is small and the base model is already good.

**Parallel to general LLM practice.** Phase 1 is "pre-training" in the LLM sense: a large general-purpose capability is established. Phase 2 is "fine-tuning": the general model is specialised to a particular deployment. The ratio is the same as for text LLMs — months of GPU hours for pre-training, days for fine-tuning.

**Why Open X-Embodiment matters.** Before Open X-Embodiment, every robotics lab collected its own data and shared nothing. Policies trained on one robot did not transfer to another. Open X-Embodiment was the "ImageNet moment" for robotics: a large, diverse, shared dataset that made it possible to train a single policy across many embodiments. OpenVLA exists because Open X-Embodiment exists.


### 5.6 RT-2 — predecessor to OpenVLA *(brief)*

RT-2 (Brohan et al., Google DeepMind, 2023) is the closed-source predecessor to OpenVLA. It is worth knowing about because it is the paper that established the core idea ("VLM + action tokens as a new vocabulary = VLA") that every subsequent open-source VLA has copied.

**Architecture.** Very similar to OpenVLA but built on PaLI-X (55B parameters) or PaLI-3 (5B), instead of LLaMA + DinoV2/SigLIP. The input is an image plus a natural-language instruction; the output is a sequence of action tokens that encode a 7-DoF robot action. The key trick — using existing rare vocabulary tokens as action bins — was introduced in RT-2.

**Why RT-2 matters historically.** It established that *Internet-scale VLMs* can be fine-tuned for robot control with minimal surgery, that the resulting models inherit visual reasoning capabilities from their pretraining (RT-2 can do "pick up the extinct animal" and it picks up the plastic dinosaur), and that the action-token trick works at scale. These three observations unlocked the entire VLA line of work that followed.

**Why RT-2 is not the open-source standard.** Closed weights, closed training data, very large. OpenVLA replaced RT-2 as the community reference because it gave everyone an open-source, reproducible, smaller version of the same ideas.


### 5.7 π₀.₅ — VLA with Open-World Generalization *(lecture p.64–65)*

Physical Intelligence (April 2025). The question the paper asks: can we train *one* VLA that we can drop into a new home and have it work without any fine-tuning? No data collection, no retraining, just "plug in and ask it to fold the laundry."

**Architecture hierarchy (lecture p.64 figure).** π₀.₅ organises the policy into three levels:

1. **High-Level.** Reads the overall task ("pick up the shirt"), looks at the scene, and reasons at the strategic level.
2. **Low-Level.** Decomposes the task into sub-goals ("approach shirt", "grasp collar", "lift"). Produces sequences of intermediate waypoints.
3. **Action Expert.** Produces the actual 7-DoF motor commands at each waypoint.

The three levels are trained jointly but execute at different time scales: the high-level runs once per task, the low-level runs a few times per task, the action expert runs at robot control rate (~50 Hz). This layering matches the natural time structure of manipulation and makes the model more controllable than a monolithic VLA.

**Training data — a deliberate mixture (lecture p.64 right panel).** π₀.₅ is trained on an unusually diverse data cocktail:

- **Multimodal web data** — image-question-answer triples from the web, teaching general visual reasoning and object recognition.
- **Subtask commands** — short instruction-action pairs that teach the low-level policy how to execute named sub-goals.
- **Object detection labels** — bounding boxes that teach the high-level to localise objects by name.
- **In-the-wild mobile robot data** — episodes collected on a mobile platform in real homes, teaching the policy to handle cluttered, unfamiliar environments.
- **In-the-wild static robot data** — similar but from stationary arms.
- **In-lab static robot data** — cleaner, more controlled episodes for fine motor skills.
- **General robot data** — Open X-Embodiment style aggregates for breadth.

The insight: no single data source teaches the model what it needs. Web data gives semantic generality. In-the-wild robot data gives robustness to real homes. In-lab data gives precise motor skills. Only by mixing all of them can the policy generalise to a new home.

**Deployment result (lecture p.64–65).** π₀.₅ is dropped — weights frozen, no fine-tuning — into homes it has never seen, with robots folding bed linens, making beds, and tidying clothing. The demonstration video (p.65) shows the policy handling messy real-world conditions, hesitating on novel objects, and recovering from its own mistakes. It is not superhuman, but it *works*, which is remarkable given that it never saw *this* home during training.

**Take-away.** π₀.₅ is the state of the art as of mid-2025 for "one VLA, multiple homes." The hierarchical architecture and the deliberate data mixture are both crucial — neither alone would produce open-world generalisation.


### 5.8 π₀.₆ — A VLA That Learns From Experience *(lecture p.66)*

Physical Intelligence (November 2025). If π₀.₅'s question was "can the VLA generalise out of the box," π₀.₆'s question is *"can the VLA keep improving from its own deployment experience?"*

**The problem π₀.₆ solves.** Every prior VLA is *frozen* after training. Deploy it, and its behaviour never improves — even if you run it for a million episodes in production, none of those episodes are used to update the weights. In contrast, humans improve with practice. A robot that has made coffee 10,000 times should be better at it than one that has made coffee 10 times.

**The approach — RL fine-tuning from autonomous episodes (lecture p.66 figure).**

1. **Start from the π₀.₆ VLA.** Pretrained on offline data like π₀.₅. This is the "base policy" that knows how to make coffee at reasonable quality.
2. **Deploy the policy autonomously.** The robot makes coffee / folds laundry / builds boxes, episode after episode, in real-world conditions.
3. **Learn a value function.** A separate critic network estimates the expected cumulative reward of each state-action pair the policy visits.
4. **Estimate advantage from the value function.** For each action the policy took, compute the *advantage* — how much better (or worse) this action was compared to the expected action at that state.
5. **Reward labels from interventions.** When a human intervenes ("oh, you spilled the coffee, let me help you clean up"), that intervention supplies a reward label for the preceding episode. These labels are sparse but informative.
6. **RL training.** Use the advantage estimates and the reward labels to fine-tune the policy via standard policy gradient methods. The action expert updates to prefer high-advantage actions and avoid low-advantage ones.

**The result.** Over time, the policy gets measurably better at each task it is practicing, without any new data collection effort from humans beyond the occasional intervention. The policy is learning *in deployment*, not just during offline training.

**Why this matters.** Every prior generation of robotics models assumed a clean separation between "training time" (lab-controlled, supervised, expensive) and "deployment time" (in the wild, frozen, free). π₀.₆ is the first serious attempt to fuse the two — the same physical robot is both running a task and collecting the data that will update its own weights. If this line of work succeeds at scale, it will change how robotics foundation models are trained and maintained: not as static snapshots, but as continuously-improving policies.


### 5.9 Challenges and Open Questions

Even with RoboPoint, ReKep, OpenVLA, and the π-series, robotics VLAs in 2025 face hard unsolved problems. The lecture does not dwell on these, but they are worth being aware of:

- **Data scarcity.** Robot data is expensive and non-stationary. A new robot, gripper, or sensor suite means most of your existing data is no longer directly usable. Open X-Embodiment helps, but it is still orders of magnitude smaller than Internet-scale text corpora.

- **Sim-to-real gap.** Simulation is cheap — you can generate billions of robot episodes in a physics engine. But policies trained purely in simulation rarely transfer cleanly to real hardware, because simulators do not capture real friction, real actuator delays, real sensor noise. Bridging this gap (via domain randomisation, real-to-sim transfer, or mixed training) is an open research problem.

- **Action tokenisation granularity.** OpenVLA bins each action component into 256 values. Too coarse → jerky, imprecise control. Too fine → explodes the vocabulary, slows the generation loop, complicates training. 256 bins is a workable compromise but not a principled optimum.

- **Safety and verification.** An LLM hallucinating a wrong word produces a typo. A VLA hallucinating a wrong action can damage hardware, drop a glass, or hurt a human. There is no mature verification layer for VLAs — no equivalent of "compile-time type checking" for robot actions. Safe deployment today relies on human monitoring and soft hardware limits.

- **Latency.** Manipulation tasks need control rates of 10–50 Hz. A 7B-parameter VLA running at 1 Hz per action decode is often too slow for reactive tasks. GQA (§3.3.6), KV caching (§3.3), and smaller backbones (PaliGemma style) are partial answers, but closing the latency gap is an ongoing engineering effort.

- **Generalisation across embodiments.** A policy trained on a Franka arm may or may not transfer to a UR5. OpenVLA and π₀.₅ make progress here by training on multi-embodiment data, but true zero-shot transfer to an unseen robot is still hit-or-miss.

These challenges are not failures of the approach — they are the frontier. Robotics VLAs are probably where vision-language models were in 2019: the architecture is established, the recipe works on some tasks, and the next few years will be about scaling data, tightening latency, and building the verification infrastructure that production deployment requires.


## Part 6 · Big Picture — Evolution of Foundation Models (W7–W9)

Six parts and ten code cells later, we can zoom out and see the whole arc from Week 7 to Week 9. This final part is a synthesis: it situates every Week 9 topic inside the three-week story and gives you a single mental diagram for "where does each technique live in a modern MLLM stack."

### 6.1 Timeline — Self-Attention → Efficient Variants → Multi-modality + Efficiency

**Week 7 — Self-Attention and the Transformer.** The foundational mechanism. Queries, keys, and values all come from the same sequence. The attention formula $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ is introduced. The whole Transformer encoder + decoder architecture is built on top of it. You also learn about positional encoding, multi-head attention, causal masking, and the basic training pipeline for a generative LM. The key limitation you discover by the end of W7: **attention is quadratic in sequence length**, which is fine for 512-token contexts but prohibitive for 8k, 32k, or 128k.

**Week 8 — Efficient Sequence Models.** The response to Week 7's quadratic cost. Two families of answers:

- **Efficient attention variants.** Linear Attention (replaces the softmax with a kernel trick to reduce complexity from $\mathcal{O}(n^2)$ to $\mathcal{O}(n)$), Sparse Attention (restricts attention to a structured subset of positions), Performer (random-feature approximation of the softmax), and Flash Attention (not an algorithmic change but a kernel-fusion trick that keeps the quadratic cost but eliminates the memory overhead). These optimisations are mostly about *training* — they make it feasible to train on long sequences.
- **State-space models.** An entirely different approach: abandon attention and use linear recurrences that can be computed in parallel via FFT-like tricks. Mamba (Selective State Space Models, S6) and Vision Mamba fall here. State-space models have near-linear complexity and competitive quality on long-context benchmarks.

**Week 9 — Multi-modality and Efficiency.** Week 9 extends the Week 7 + 8 foundations in two parallel directions.

*Multi-modality:*
- **Cross-Attention (§2.6)** — the single mechanism that lets one sequence (text) read information from another (image). An almost trivial variation of self-attention (Q from one source, K/V from another) that turns out to be the foundation of every VLM connector.
- **Connectors (§2.5)** — the module that wires a vision encoder to an LLM. Token-level fusion (LLaVA, Q-Former) or feature-level fusion (Flamingo, CogVLM). The architectural creativity of modern MLLMs lives mostly in this one box.
- **Multi-modal generation (§2.8)** — extending the LLM's output vocabulary to include visual tokens (SEED, SEED-X, VAR) or coupling the LLM with diffusion (DiffusionGPT, Transfusion). The frontier of "one model that reads and writes everything."
- **VLA (§5)** — extending the same trick to robot actions. OpenVLA is "SEED-X for motor commands."

*Efficiency:*
- **LoRA (§3.2)** — train a few million low-rank parameters on top of a frozen backbone instead of fine-tuning the whole thing. Training-time efficiency.
- **KV Cache (§3.3)** — avoid re-computing K and V across decode steps. Inference-time efficiency. Non-negotiable in production.
- **GQA (§3.3.6)** — share K/V heads across Q heads to shrink the KV cache by 4×. Now the default in LLaMA-2 70B, Mistral, Qwen.
- **MoE (§4)** — activate only a fraction of the expert parameters per token. Architectural efficiency: large total model, small per-token compute.

The arc: **Week 7 built the primitive, Week 8 optimised its cost, Week 9 extended it across modalities and added the efficiency layer that makes production deployment possible.** Every frontier 2024–2025 MLLM and VLA can be described as "pick one modality mechanism from Week 9's first half and one efficiency mechanism from Week 9's second half, then train."


### 6.2 Synthesis — where each technique sits in a modern MLLM stack

Here is a deliberately simplified block diagram of a modern production MLLM — say, LLaVA-1.5 served via vLLM. Every box maps to a Week 9 section number.

```
                            ┌────────────────────────────────────┐
                            │                                    │
   image ──> Vision encoder │  Connector (§2.5)                  │
             (CLIP / SigLIP │  - MLP projector (LLaVA, §2.5.2)   │──┐
              / DinoV2)     │  - OR Q-Former (BLIP-2, §2.5.2)    │  │
                            │  - OR gated cross-attn (Flamingo, │  │
                            │    §2.5.3, §2.7.3)                 │  │
                            └────────────────────────────────────┘  │
                                                                    │
                                                                    v
                                                     ┌────────────────────────────┐
                                                     │  LLM Decoder Stack         │
                                                     │                            │
                                                     │  - LoRA (§3.2) wraps W_Q,  │
                                                     │    W_V for PEFT fine-tune  │
                                                     │                            │
                                                     │  - MoE (§4) replaces MLP   │
                                                     │    blocks in some layers   │
                                                     │                            │
                                                     │  - GQA (§3.3.6) inside     │
                                                     │    every self-attention    │
                                                     │                            │
                                                     └─────────────┬──────────────┘
                                                                   │
                                                                   v
                                                     ┌────────────────────────────┐
                                                     │  KV Cache (§3.3)           │
                                                     │  - paged (§3.3.10)         │
                                                     │  - int8 quantised (§3.3.8) │
                                                     │  - attention-sink eviction │
                                                     │    (§3.3.9)                │
                                                     └─────────────┬──────────────┘
                                                                   │
                                                                   v
                                                         text output
                                                           (or visual tokens via
                                                            §2.8 generation,
                                                            or action tokens via
                                                            §5.4 OpenVLA)
```

**Read the diagram as follows.** An image and a text prompt enter the system. The vision encoder (CLIP / SigLIP / DinoV2, §2.4) turns the image into visual features. The **connector** (§2.5 — the most creative box in any MLLM paper) maps those features into something the LLM can consume: either as token-level visual tokens that are concatenated with text (LLaVA-style, §2.5.2) or as feature-level injections via gated cross-attention (Flamingo-style, §2.5.3).

The **LLM decoder stack** (§1.2, §1.3, §1.6) consumes the mixed visual-and-text token sequence. Inside the decoder, three Week 9 efficiency mechanisms are active in parallel:

- LoRA (§3.2) may wrap the attention projections, making the whole stack fine-tunable with ~0.06% of the trainable parameters.
- MoE (§4) may replace the MLP block in some layers, giving the model enormous total capacity while only activating a fraction of the experts per token.
- GQA (§3.3.6) is inside every self-attention layer, reducing the KV-cache size by a factor of 4 at no significant quality cost.

The **KV cache** (§3.3) sits next to the decoder stack, absorbing the K and V projections across decode steps. The cache itself is paged (§3.3.10), quantised to int8 (§3.3.8), and managed with an attention-sink-aware eviction policy (§3.3.9) — the three tricks that together make vLLM-style serving possible.

The **output** is a sequence of text tokens by default. In §2.8-style generative MLLMs (SEED-X, VAR, Transfusion), the output vocabulary is extended with visual tokens so the model can produce images. In §5.4-style VLAs (OpenVLA), the output vocabulary is extended with action tokens so the model can produce robot actions. Both are special cases of the same "vocabulary extension" trick: reassign rare tokens to new semantic roles, fine-tune to make the LLM emit them in the right contexts, decode them back into the target modality.

**That is the entire Week 9 syllabus in one diagram.** Every technique you have learned in this notebook is a node (or an edge) in this picture. The next time you read a new MLLM or VLA paper, start by locating its contribution on this diagram. Usually it is one node — a better connector, a better vision encoder, a smaller LLM, a more efficient KV cache — and you can read the paper asking only "what is different at this node?"


### 6.3 Further reading

A minimal reading list, grouped by Part. These are the papers that most directly underlie the content of this notebook.

**Part 1 — LLMs**
- Vaswani et al. "Attention is All You Need." NeurIPS 2017. *(The original Transformer.)*
- Brown et al. "Language Models are Few-Shot Learners." NeurIPS 2020. *(GPT-3.)*
- Ouyang et al. "Training language models to follow instructions with human feedback." NeurIPS 2022. *(InstructGPT, RLHF.)*
- Rafailov et al. "Direct Preference Optimization: Your Language Model is Secretly a Reward Model." NeurIPS 2023. *(DPO.)*

**Part 2 — MLLMs**
- Radford et al. "Learning Transferable Visual Models From Natural Language Supervision." ICML 2021. *(CLIP.)*
- Alayrac et al. "Flamingo: a Visual Language Model for Few-Shot Learning." NeurIPS 2022.
- Li et al. "BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and Large Language Models." ICML 2023. *(Q-Former.)*
- Liu et al. "Visual Instruction Tuning." NeurIPS 2023. *(LLaVA.)*
- Ge et al. "Making LLaMA SEE and Draw with SEED Tokenizer." ICLR 2024. *(SEED.)*
- Ge et al. "SEED-X: Multimodal Models with Unified Multi-granularity Comprehension and Generation." 2024.
- Tian et al. "Visual Autoregressive Modeling: Scalable Image Generation via Next-Scale Prediction." NeurIPS 2024. *(VAR.)*

**Part 3 — Efficiency**
- Hu et al. "LoRA: Low-Rank Adaptation of Large Language Models." ICLR 2022.
- Shazeer. "Fast Transformer Decoding: One Write-Head is All You Need." 2019. *(MQA.)*
- Ainslie et al. "GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints." EMNLP 2023.
- Kwon et al. "Efficient Memory Management for Large Language Model Serving with PagedAttention." SOSP 2023. *(vLLM.)*
- Xiao et al. "Efficient Streaming Language Models with Attention Sinks." ICLR 2024.

**Part 4 — Mixture of Experts**
- Shazeer et al. "Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer." ICLR 2017.
- Fedus et al. "Switch Transformer: Scaling to Trillion Parameter Models with Simple and Efficient Sparsity." JMLR 2022.
- Lin et al. "MoE-LLaVA: Mixture of Experts for Large Vision-Language Models." 2024.
- Pan et al. "Dense Training, Sparse Inference: Rethinking Training of Mixture-of-Experts Language Models." 2024. *(DS-MoE.)*

**Part 5 — Robotics VLAs**
- Brohan et al. "RT-2: Vision-Language-Action Models Transfer Web Knowledge to Robotic Control." CoRL 2023.
- Kim et al. "OpenVLA: An Open-Source Vision-Language-Action Model." CoRL 2024.
- "RoboPoint: A Vision-Language Model for Spatial Affordance Prediction for Robotics." CoRL 2024.
- "ReKep: Spatio-Temporal Reasoning of Relational Keypoint Constraints for Robotic Manipulation." CoRL 2024.
- Physical Intelligence. "π₀.₅: a Vision-Language-Action Model with Open-World Generalization." 2025.
- Physical Intelligence. "π₀.₆: a VLA that Learns from Experience." 2025.

**End of self-study material.** You now have a complete, independent reference for Week 9 — every topic in the 67-slide lecture covered at depth, ten from-scratch code cells for the core mechanisms, and a synthesis diagram that ties everything together. The tutorial will build on §2.6 (Cross-Attention) and §3.3 (KV Cache) specifically; everything else in this notebook is yours to study at your own pace.
